# CEREBRO PoC — 06.5 Reusable Pipeline Extraction

**Stage:** Controlled Refactoring  
**Purpose:** Convert validated notebook logic into reusable CEREBRO capabilities without changing behaviour.

---

## Objective

Steps 00–06 validated the core CEREBRO knowledge pipeline.

This stage extracts validated processing logic from the experimental notebooks into reusable modules under `poc/src/`.

The notebooks remain the experimental and regression reference.

### Target Architecture

Notebook Experiments  
↓  
Reusable CEREBRO Capabilities (`poc/src`)  
↓  
Application / Future Experiments / Automated Tests

### Refactoring Rule

**Existing validated behaviour is authoritative.**

This stage must not introduce:

- new algorithms,
- new models,
- new thresholds,
- schema redesign,
- identifier changes,
- provenance changes,
- new AI behaviour,
- architecture redesign.

The objective is:

**Extract → Reuse → Compare → Verify**

not:

**Rewrite → Improve → Replace**

### Regression Requirement

After extraction:

**Steps 00–06 must remain reproducible and semantically equivalent.**

The following must remain stable where applicable:

- `ART-*` artifact identity
- `KF-*` knowledge fragment identity
- source provenance
- source locations
- trusted relationships
- candidate relationships
- temporal semantics
- embedding model
- Assisted Recollection evidence
- CEREBRO Knowledge Experience Contract

If extraction changes validated behaviour unexpectedly, the refactoring fails.

### 1 — Baseline Inventory

In [7]:
from pathlib import Path
import json

# ---------------------------------------------------------
# CEREBRO 06.5 — Regression Baseline
# ---------------------------------------------------------

repo_root = Path.cwd().parents[1]

paths = {
    "registered_artifact":
        repo_root
        / "poc/data/processed/artifacts/ART-0001.json",

    "knowledge_model":
        repo_root
        / "poc/data/processed/knowledge/KM-0001.json",

    "semantic_candidates":
        repo_root
        / "poc/data/processed/knowledge/KM-0001-semantic-candidates.json",

    "experience_contract":
        repo_root
        / "poc/outputs/integration/CEREBRO-KNOWLEDGE-EXPERIENCE-v0.1.json",

    "source_artifact":
        repo_root
        / "poc/data/raw/text/benchmark_001.txt",
}

print("CEREBRO — 06.5 Baseline Inventory")
print("=" * 50)

all_present = True

for name, path in paths.items():

    exists = path.exists()

    print(
        f"{name:22} : "
        f"{'✓ FOUND' if exists else '✗ MISSING'}"
    )

    if not exists:
        print("   ", path)
        all_present = False

assert all_present, (
    "06.5 cannot continue: "
    "required validated outputs are missing."
)

print("\n✓ All required baseline artifacts found")

CEREBRO — 06.5 Baseline Inventory
registered_artifact    : ✓ FOUND
knowledge_model        : ✓ FOUND
semantic_candidates    : ✓ FOUND
experience_contract    : ✓ FOUND
source_artifact        : ✓ FOUND

✓ All required baseline artifacts found


### 1a - Select Artifact

In [5]:
import ipywidgets as widgets
from IPython.display import display

# ---------------------------------------------------------
# CEREBRO 06.5 — Dynamic Artifact Selection
# ---------------------------------------------------------

uploader = widgets.FileUpload(
    accept="",          # Accept anything — CEREBRO identifies it
    multiple=False,
    description="Select Artifact"
)

display(uploader)

FileUpload(value=(), description='Select Artifact')

### 1b — Load and Fingerprint Baseline

In [10]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Capture Selected Artifact
# ---------------------------------------------------------

if not uploader.value:
    raise RuntimeError(
        "No artifact selected. "
        "Use the Select Artifact button in Cell 1A first."
    )

uploaded = uploader.value

# ipywidgets 8.x
if isinstance(uploaded, tuple):
    uploaded_file = uploaded[0]

# Compatibility with older ipywidgets
elif isinstance(uploaded, dict):
    uploaded_file = next(iter(uploaded.values()))

else:
    raise TypeError(
        f"Unsupported uploader value type: {type(uploaded)}"
    )

filename = uploaded_file["name"]
file_bytes = bytes(uploaded_file["content"])

print("CEREBRO — Artifact Received")
print("=" * 60)
print(f"Filename : {filename}")
print(f"Size     : {len(file_bytes):,} bytes")

print("\n✓ Artifact captured as raw bytes")
print("✓ No artifact type assumed")
print("✓ Ready for deterministic identification")

CEREBRO — Artifact Received
Filename : benchmark_001.txt
Size     : 294 bytes

✓ Artifact captured as raw bytes
✓ No artifact type assumed
✓ Ready for deterministic identification


### 2 — Deterministic Artifact Identification

In [11]:
import mimetypes
import hashlib


# ---------------------------------------------------------
# CEREBRO 06.5 — Deterministic Artifact Identification
# ---------------------------------------------------------

def detect_file_signature(content: bytes):
    """
    Identify common artifact types using magic-byte signatures.
    Does not trust filename or extension.
    """

    signatures = [
        (b"%PDF-", "application/pdf", "pdf"),
        (b"\x89PNG\r\n\x1a\n", "image/png", "image"),
        (b"\xff\xd8\xff", "image/jpeg", "image"),
        (b"GIF87a", "image/gif", "image"),
        (b"GIF89a", "image/gif", "image"),
        (b"RIFF", "riff-container", "container"),
        (b"ID3", "audio/mpeg", "audio"),
        (b"PK\x03\x04", "zip-container", "container"),
    ]

    for signature, detected_type, modality in signatures:
        if content.startswith(signature):
            return detected_type, modality

    return None, None


# Filename-based MIME guess — advisory only
extension_mime, _ = mimetypes.guess_type(filename)

# Byte-based identification
signature_type, signature_modality = detect_file_signature(file_bytes)

# Integrity
sha256 = hashlib.sha256(file_bytes).hexdigest()


print("CEREBRO — Artifact Identification")
print("=" * 60)

print(f"Filename             : {filename}")
print(f"Extension MIME guess : {extension_mime or 'Unknown'}")
print(f"Byte signature       : {signature_type or 'No known binary signature'}")
print(f"Initial modality     : {signature_modality or 'Undetermined'}")
print(f"Size                 : {len(file_bytes):,} bytes")
print(f"SHA-256              : {sha256}")

print("\n✓ Identification performed without trusting extension")

CEREBRO — Artifact Identification
Filename             : benchmark_001.txt
Extension MIME guess : text/plain
Byte signature       : No known binary signature
Initial modality     : Undetermined
Size                 : 294 bytes
SHA-256              : 7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832

✓ Identification performed without trusting extension


### 3 - Text Content Validation

In [13]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Text Content Validation
# ---------------------------------------------------------

def validate_text_content(content: bytes):
    """
    Determine whether raw bytes represent plausible UTF-8 text.

    Returns evidence rather than relying on filename extension.
    """

    result = {
        "is_utf8": False,
        "is_text": False,
        "printable_ratio": 0.0,
        "null_bytes": 0,
        "decoded_text": None,
        "reason": None,
    }

    # Null bytes are a strong binary indicator
    result["null_bytes"] = content.count(b"\x00")

    try:
        decoded = content.decode("utf-8")
        result["is_utf8"] = True
        result["decoded_text"] = decoded

    except UnicodeDecodeError:
        result["reason"] = "Content is not valid UTF-8."
        return result

    if not decoded:
        result["reason"] = "Artifact contains no text."
        return result

    # Count printable characters.
    # Whitespace used in normal text is considered valid.
    printable_count = sum(
        1
        for char in decoded
        if char.isprintable() or char in "\n\r\t"
    )

    result["printable_ratio"] = printable_count / len(decoded)

    # Conservative deterministic rule for this PoC
    result["is_text"] = (
        result["is_utf8"]
        and result["null_bytes"] == 0
        and result["printable_ratio"] >= 0.95
    )

    if result["is_text"]:
        result["reason"] = "Valid UTF-8 with high printable-text ratio."
    else:
        result["reason"] = "Content does not satisfy text validation rules."

    return result


# ---------------------------------------------------------
# Run validation
# ---------------------------------------------------------

text_validation = validate_text_content(file_bytes)

print("CEREBRO — Text Content Validation")
print("=" * 60)

print(f"Valid UTF-8       : {text_validation['is_utf8']}")
print(f"Printable ratio   : {text_validation['printable_ratio']:.4f}")
print(f"Null bytes        : {text_validation['null_bytes']}")
print(f"Validated as text : {text_validation['is_text']}")
print(f"Reason            : {text_validation['reason']}")


# ---------------------------------------------------------
# Initial Modality Resolution
# ---------------------------------------------------------

if signature_modality:
    detected_modality = signature_modality

elif text_validation["is_text"]:
    detected_modality = "text"

else:
    detected_modality = "unknown"


print(f"\nDetected modality : {detected_modality}")

assert detected_modality == "text", (
    "Regression artifact benchmark_001.txt "
    "was expected to resolve as text."
)

print("\n✓ Content-based modality detection PASS")

CEREBRO — Text Content Validation
Valid UTF-8       : True
Printable ratio   : 1.0000
Null bytes        : 0
Validated as text : True
Reason            : Valid UTF-8 with high printable-text ratio.

Detected modality : text

✓ Content-based modality detection PASS


### 4 — Extension vs Content Validation

In [14]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Extension vs Content Validation
# ---------------------------------------------------------

def expected_modality_from_filename(name: str):
    """
    Advisory classification based only on filename extension.
    This must never override content-based identification.
    """

    extension = name.lower().rsplit(".", 1)[-1] if "." in name else ""

    mapping = {
        "txt": "text",
        "pdf": "pdf",

        "png": "image",
        "jpg": "image",
        "jpeg": "image",
        "gif": "image",

        "wav": "audio",
        "mp3": "audio",

        "mp4": "video",

        "pptx": "office",
        "docx": "office",
    }

    return mapping.get(extension, "unknown")


def validate_filename_consistency(
    name: str,
    detected_modality: str
):
    declared_modality = expected_modality_from_filename(name)

    if declared_modality == "unknown":
        status = "UNVERIFIED"

    elif detected_modality == "unknown":
        status = "UNVERIFIED"

    elif declared_modality == detected_modality:
        status = "MATCH"

    else:
        status = "MISMATCH"

    return {
        "filename": name,
        "declared_modality": declared_modality,
        "detected_modality": detected_modality,
        "status": status,
    }


# ---------------------------------------------------------
# Test 1 — Real filename
# ---------------------------------------------------------

real_test = validate_filename_consistency(
    filename,
    detected_modality
)


# ---------------------------------------------------------
# Test 2 — Deliberately misleading filename
#
# SAME CONTENT.
# We are NOT modifying the original artifact.
# ---------------------------------------------------------

fake_filename = "benchmark_001.pdf"

mismatch_test = validate_filename_consistency(
    fake_filename,
    detected_modality
)


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("CEREBRO — Artifact Consistency Validation")
print("=" * 60)

print("\nTEST 1 — Original Filename")
print(f"Filename          : {real_test['filename']}")
print(f"Declared modality : {real_test['declared_modality']}")
print(f"Detected modality : {real_test['detected_modality']}")
print(f"Status            : {real_test['status']}")

print("\nTEST 2 — Misleading Filename")
print(f"Filename          : {mismatch_test['filename']}")
print(f"Declared modality : {mismatch_test['declared_modality']}")
print(f"Detected modality : {mismatch_test['detected_modality']}")
print(f"Status            : {mismatch_test['status']}")


# ---------------------------------------------------------
# Regression Assertions
# ---------------------------------------------------------

assert real_test["status"] == "MATCH", (
    "Known benchmark filename should match detected content."
)

assert mismatch_test["status"] == "MISMATCH", (
    "CEREBRO failed to detect misleading filename."
)

print("\n✓ Correct filename accepted")
print("✓ Misleading filename detected")
print("✓ Extension is advisory, not authoritative")
print("✓ Artifact consistency validation PASS")

CEREBRO — Artifact Consistency Validation

TEST 1 — Original Filename
Filename          : benchmark_001.txt
Declared modality : text
Detected modality : text
Status            : MATCH

TEST 2 — Misleading Filename
Filename          : benchmark_001.pdf
Declared modality : pdf
Detected modality : text
Status            : MISMATCH

✓ Correct filename accepted
✓ Misleading filename detected
✓ Extension is advisory, not authoritative
✓ Artifact consistency validation PASS


### 5 — Artifact Inspection Contract

In [15]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Artifact Inspection Contract
# ---------------------------------------------------------

from datetime import datetime, timezone


def build_artifact_inspection(
    filename: str,
    content: bytes,
    extension_mime: str | None,
    signature_type: str | None,
    detected_modality: str,
    text_validation: dict,
):
    """
    Build the canonical deterministic inspection result.

    IMPORTANT:
    This represents inspection evidence only.
    It is NOT artifact registration and does not create trusted
    knowledge.
    """

    consistency = validate_filename_consistency(
        filename,
        detected_modality
    )

    return {
        "filename": filename,

        "file": {
            "size_bytes": len(content),
            "sha256": hashlib.sha256(content).hexdigest(),
        },

        "identification": {
            "extension_mime_guess": extension_mime,
            "signature_type": signature_type,
            "detected_modality": detected_modality,
            "filename_consistency": consistency["status"],
        },

        "content_validation": {
            "is_utf8": text_validation["is_utf8"],
            "is_text": text_validation["is_text"],
            "printable_ratio": text_validation["printable_ratio"],
            "null_bytes": text_validation["null_bytes"],
            "reason": text_validation["reason"],
        },

        "provenance": {
            "method": "deterministic_artifact_inspection",
            "ai_used": False,
            "human_confirmed": False,
            "inspected_at": datetime.now(
                timezone.utc
            ).isoformat(),
        },

        "status": "INSPECTED",
    }


artifact_inspection = build_artifact_inspection(
    filename=filename,
    content=file_bytes,
    extension_mime=extension_mime,
    signature_type=signature_type,
    detected_modality=detected_modality,
    text_validation=text_validation,
)


print("CEREBRO — Artifact Inspection")
print("=" * 60)

print(json.dumps(
    artifact_inspection,
    indent=2,
    ensure_ascii=False
))

CEREBRO — Artifact Inspection
{
  "filename": "benchmark_001.txt",
  "file": {
    "size_bytes": 294,
    "sha256": "7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832"
  },
  "identification": {
    "extension_mime_guess": "text/plain",
    "signature_type": null,
    "detected_modality": "text",
    "filename_consistency": "MATCH"
  },
  "content_validation": {
    "is_utf8": true,
    "is_text": true,
    "printable_ratio": 1.0,
    "null_bytes": 0,
    "reason": "Valid UTF-8 with high printable-text ratio."
  },
  "provenance": {
    "method": "deterministic_artifact_inspection",
    "ai_used": false,
    "human_confirmed": false,
    "inspected_at": "2026-09-24T07:34:37.063823+00:00"
  },
  "status": "INSPECTED"
}


### 5a - Contract Validation

In [17]:
assert artifact_inspection["filename"] == filename
assert (
    artifact_inspection["file"]["sha256"]
    == hashlib.sha256(file_bytes).hexdigest()
)
assert (
    artifact_inspection["identification"]["detected_modality"]
    == "text"
)
assert (
    artifact_inspection["identification"]["filename_consistency"]
    == "MATCH"
)
assert artifact_inspection["content_validation"]["is_text"] is True
assert artifact_inspection["provenance"]["ai_used"] is False
assert artifact_inspection["provenance"]["human_confirmed"] is False
assert artifact_inspection["status"] == "INSPECTED"

print("\n✓ Deterministic inspection contract created")
print("✓ Source checksum preserved")
print("✓ Modality evidence preserved")
print("✓ Filename/content consistency preserved")
print("✓ No AI used")
print("✓ No human approval implied")
print("✓ Artifact remains unregistered")
print("✓ Artifact inspection contract PASS")


✓ Deterministic inspection contract created
✓ Source checksum preserved
✓ Modality evidence preserved
✓ Filename/content consistency preserved
✓ No AI used
✓ No human approval implied
✓ Artifact remains unregistered
✓ Artifact inspection contract PASS


### 6a — Create reusable inspection.py

In [18]:
from pathlib import Path

# ---------------------------------------------------------
# CEREBRO 06.5 — Extract Reusable Inspection Capability
# ---------------------------------------------------------

inspection_module = repo_root / "poc/src/inspection.py"

inspection_code = '''\
"""
CEREBRO deterministic artifact inspection.

This module identifies and validates an artifact using deterministic
evidence before AI processing.

No AI inference.
No artifact registration.
No knowledge construction.
"""

from datetime import datetime, timezone
import hashlib
import mimetypes


def detect_file_signature(content: bytes):
    """
    Identify common artifact types using byte signatures.

    Filename and extension are not trusted as authoritative evidence.
    """

    signatures = [
        (b"%PDF-", "application/pdf", "pdf"),
        (b"\\x89PNG\\r\\n\\x1a\\n", "image/png", "image"),
        (b"\\xff\\xd8\\xff", "image/jpeg", "image"),
        (b"GIF87a", "image/gif", "image"),
        (b"GIF89a", "image/gif", "image"),
        (b"RIFF", "riff-container", "container"),
        (b"ID3", "audio/mpeg", "audio"),
        (b"PK\\x03\\x04", "zip-container", "container"),
    ]

    for signature, detected_type, modality in signatures:
        if content.startswith(signature):
            return detected_type, modality

    return None, None


def validate_text_content(content: bytes):
    """
    Determine whether raw bytes represent plausible UTF-8 text.
    """

    result = {
        "is_utf8": False,
        "is_text": False,
        "printable_ratio": 0.0,
        "null_bytes": 0,
        "decoded_text": None,
        "reason": None,
    }

    result["null_bytes"] = content.count(b"\\x00")

    try:
        decoded = content.decode("utf-8")
        result["is_utf8"] = True
        result["decoded_text"] = decoded

    except UnicodeDecodeError:
        result["reason"] = "Content is not valid UTF-8."
        return result

    if not decoded:
        result["reason"] = "Artifact contains no text."
        return result

    printable_count = sum(
        1
        for char in decoded
        if char.isprintable() or char in "\\n\\r\\t"
    )

    result["printable_ratio"] = (
        printable_count / len(decoded)
    )

    result["is_text"] = (
        result["is_utf8"]
        and result["null_bytes"] == 0
        and result["printable_ratio"] >= 0.95
    )

    if result["is_text"]:
        result["reason"] = (
            "Valid UTF-8 with high printable-text ratio."
        )
    else:
        result["reason"] = (
            "Content does not satisfy text validation rules."
        )

    return result


def expected_modality_from_filename(name: str):
    """
    Return advisory modality based on extension.
    """

    extension = (
        name.lower().rsplit(".", 1)[-1]
        if "." in name
        else ""
    )

    mapping = {
        "txt": "text",
        "pdf": "pdf",

        "png": "image",
        "jpg": "image",
        "jpeg": "image",
        "gif": "image",

        "wav": "audio",
        "mp3": "audio",

        "mp4": "video",

        "pptx": "office",
        "docx": "office",
    }

    return mapping.get(extension, "unknown")


def validate_filename_consistency(
    name: str,
    detected_modality: str
):
    """
    Compare advisory filename modality with content-derived modality.
    """

    declared_modality = expected_modality_from_filename(name)

    if declared_modality == "unknown":
        status = "UNVERIFIED"

    elif detected_modality == "unknown":
        status = "UNVERIFIED"

    elif declared_modality == detected_modality:
        status = "MATCH"

    else:
        status = "MISMATCH"

    return {
        "filename": name,
        "declared_modality": declared_modality,
        "detected_modality": detected_modality,
        "status": status,
    }


def inspect_artifact(
    filename: str,
    content: bytes
):
    """
    Perform deterministic CEREBRO artifact inspection.

    Input:
        filename
        raw artifact bytes

    Output:
        deterministic inspection contract
    """

    extension_mime, _ = mimetypes.guess_type(filename)

    signature_type, signature_modality = (
        detect_file_signature(content)
    )

    text_validation = validate_text_content(content)

    if signature_modality:
        detected_modality = signature_modality

    elif text_validation["is_text"]:
        detected_modality = "text"

    else:
        detected_modality = "unknown"

    consistency = validate_filename_consistency(
        filename,
        detected_modality
    )

    return {
        "filename": filename,

        "file": {
            "size_bytes": len(content),
            "sha256": hashlib.sha256(content).hexdigest(),
        },

        "identification": {
            "extension_mime_guess": extension_mime,
            "signature_type": signature_type,
            "detected_modality": detected_modality,
            "filename_consistency": consistency["status"],
        },

        "content_validation": {
            "is_utf8": text_validation["is_utf8"],
            "is_text": text_validation["is_text"],
            "printable_ratio": text_validation["printable_ratio"],
            "null_bytes": text_validation["null_bytes"],
            "reason": text_validation["reason"],
        },

        "provenance": {
            "method": "deterministic_artifact_inspection",
            "ai_used": False,
            "human_confirmed": False,
            "inspected_at": datetime.now(
                timezone.utc
            ).isoformat(),
        },

        "status": "INSPECTED",
    }
'''

inspection_module.write_text(
    inspection_code,
    encoding="utf-8"
)

print("CEREBRO — Reusable Capability Extraction")
print("=" * 60)
print(f"Created : {inspection_module}")
print("\n✓ inspection.py created")

CEREBRO — Reusable Capability Extraction
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/inspection.py

✓ inspection.py created


### 6B — Import reusable capability

In [19]:
import sys
import importlib

src_path = repo_root / "poc/src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import inspection
importlib.reload(inspection)

print("✓ CEREBRO inspection capability imported")

✓ CEREBRO inspection capability imported


6C — Run the same artifact through reusable code

In [20]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Reusable Inspection Regression
# ---------------------------------------------------------

reusable_inspection = inspection.inspect_artifact(
    filename=filename,
    content=file_bytes
)

print("CEREBRO — Reusable Inspection Result")
print("=" * 60)

print(json.dumps(
    reusable_inspection,
    indent=2,
    ensure_ascii=False
))

CEREBRO — Reusable Inspection Result
{
  "filename": "benchmark_001.txt",
  "file": {
    "size_bytes": 294,
    "sha256": "7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832"
  },
  "identification": {
    "extension_mime_guess": "text/plain",
    "signature_type": null,
    "detected_modality": "text",
    "filename_consistency": "MATCH"
  },
  "content_validation": {
    "is_utf8": true,
    "is_text": true,
    "printable_ratio": 1.0,
    "null_bytes": 0,
    "reason": "Valid UTF-8 with high printable-text ratio."
  },
  "provenance": {
    "method": "deterministic_artifact_inspection",
    "ai_used": false,
    "human_confirmed": false,
    "inspected_at": "2026-09-24T07:41:27.213496+00:00"
  },
  "status": "INSPECTED"
}


### 6D — Compare old vs reusable behavior

In [21]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Inspection Regression Test
# ---------------------------------------------------------

assert reusable_inspection["filename"] == artifact_inspection["filename"]

assert reusable_inspection["file"] == artifact_inspection["file"]

assert (
    reusable_inspection["identification"]
    == artifact_inspection["identification"]
)

assert (
    reusable_inspection["content_validation"]
    == artifact_inspection["content_validation"]
)

assert (
    reusable_inspection["provenance"]["method"]
    == artifact_inspection["provenance"]["method"]
)

assert reusable_inspection["provenance"]["ai_used"] is False

assert reusable_inspection["provenance"]["human_confirmed"] is False

assert reusable_inspection["status"] == artifact_inspection["status"]


print("CEREBRO — Inspection Refactor Regression")
print("=" * 60)

print("✓ Filename preserved")
print("✓ SHA-256 preserved")
print("✓ File size preserved")
print("✓ Modality detection preserved")
print("✓ MIME/signature evidence preserved")
print("✓ Text validation preserved")
print("✓ Filename consistency preserved")
print("✓ Provenance semantics preserved")
print("✓ No AI introduced")
print("✓ No registration introduced")

print("\n✓ REUSABLE INSPECTION CAPABILITY PASS")

CEREBRO — Inspection Refactor Regression
✓ Filename preserved
✓ SHA-256 preserved
✓ File size preserved
✓ Modality detection preserved
✓ MIME/signature evidence preserved
✓ Text validation preserved
✓ Filename consistency preserved
✓ Provenance semantics preserved
✓ No AI introduced
✓ No registration introduced

✓ REUSABLE INSPECTION CAPABILITY PASS


### 7A — Create reusable extraction.py

In [22]:
from pathlib import Path

# ---------------------------------------------------------
# CEREBRO 06.5 — Reusable Extraction Capability
# ---------------------------------------------------------

extraction_module = repo_root / "poc/src/extraction.py"

extraction_code = '''\
"""
CEREBRO deterministic content extraction.

Transforms an inspected artifact into a provenance-aware
representation suitable for downstream CEREBRO processing.

No AI inference.
No metadata enrichment.
No artifact registration.
No knowledge construction.
"""


def extract_text_artifact(
    filename: str,
    content: bytes,
    inspection: dict
):
    """
    Extract UTF-8 text from an artifact already validated as text.
    """

    detected_modality = (
        inspection
        .get("identification", {})
        .get("detected_modality")
    )

    if detected_modality != "text":
        raise ValueError(
            "extract_text_artifact requires "
            f"detected_modality='text', got {detected_modality!r}"
        )

    if not inspection.get(
        "content_validation", {}
    ).get("is_text", False):
        raise ValueError(
            "Artifact did not pass deterministic text validation."
        )

    source_text = content.decode("utf-8")

    return {
        "filename": filename,

        "modality": "text",

        "representation": {
            "type": "text",
            "content": source_text,
            "character_count": len(source_text),
        },

        "source": {
            "sha256": inspection["file"]["sha256"],
            "size_bytes": inspection["file"]["size_bytes"],
        },

        "provenance": {
            "method": "utf8_text_extraction",
            "source_representation": "original_artifact",
            "ai_used": False,
            "human_confirmed": False,
        },

        "status": "EXTRACTED",
    }
'''

extraction_module.write_text(
    extraction_code,
    encoding="utf-8"
)

print("CEREBRO — Extraction Capability")
print("=" * 60)
print(f"Created : {extraction_module}")
print()
print("✓ extraction.py created")

CEREBRO — Extraction Capability
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/extraction.py

✓ extraction.py created


### 7B — Import and run it

In [23]:
import extraction
importlib.reload(extraction)

extracted_artifact = extraction.extract_text_artifact(
    filename=filename,
    content=file_bytes,
    inspection=reusable_inspection
)

print("CEREBRO — Extracted Artifact")
print("=" * 60)

print(f"Filename        : {extracted_artifact['filename']}")
print(f"Modality        : {extracted_artifact['modality']}")
print(f"Representation  : {extracted_artifact['representation']['type']}")
print(f"Characters      : {extracted_artifact['representation']['character_count']}")
print(f"Method          : {extracted_artifact['provenance']['method']}")
print(f"AI used         : {extracted_artifact['provenance']['ai_used']}")
print(f"Status          : {extracted_artifact['status']}")

CEREBRO — Extracted Artifact
Filename        : benchmark_001.txt
Modality        : text
Representation  : text
Characters      : 294
Method          : utf8_text_extraction
AI used         : False
Status          : EXTRACTED


### 7C — Regression against the actual uploaded bytes

In [24]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Extraction Regression
# ---------------------------------------------------------

expected_text = file_bytes.decode("utf-8")

assert (
    extracted_artifact["representation"]["content"]
    == expected_text
)

assert (
    extracted_artifact["representation"]["character_count"]
    == len(expected_text)
)

assert (
    extracted_artifact["source"]["sha256"]
    == reusable_inspection["file"]["sha256"]
)

assert extracted_artifact["modality"] == "text"

assert (
    extracted_artifact["provenance"]["source_representation"]
    == "original_artifact"
)

assert extracted_artifact["provenance"]["ai_used"] is False

assert extracted_artifact["status"] == "EXTRACTED"


print("CEREBRO — Extraction Regression")
print("=" * 60)

print("✓ Exact source text preserved")
print("✓ Character count preserved")
print("✓ Source checksum propagated")
print("✓ Source lineage preserved")
print("✓ No AI introduced")
print("✓ Artifact remains unregistered")

print("\n✓ REUSABLE EXTRACTION CAPABILITY PASS")

CEREBRO — Extraction Regression
✓ Exact source text preserved
✓ Character count preserved
✓ Source checksum propagated
✓ Source lineage preserved
✓ No AI introduced
✓ Artifact remains unregistered

✓ REUSABLE EXTRACTION CAPABILITY PASS


### 8A — Create metadata.py

In [25]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Reusable Metadata Capability
# ---------------------------------------------------------

metadata_module = repo_root / "poc/src/metadata.py"

metadata_code = '''\
"""
CEREBRO deterministic metadata extraction.

Produces metadata and prefill evidence from an extracted artifact
without AI inference.

AI enrichment belongs to a later capability.
Human confirmation belongs to the review stage.
"""

from pathlib import Path


def extract_text_metadata(extracted_artifact: dict):
    """
    Produce deterministic/extracted metadata for a text artifact.

    This deliberately avoids semantic inference.
    """

    filename = extracted_artifact["filename"]
    representation = extracted_artifact["representation"]

    if extracted_artifact["modality"] != "text":
        raise ValueError(
            "extract_text_metadata currently supports "
            "validated text artifacts only."
        )

    source_text = representation["content"]

    # Filename stem is only a deterministic title candidate.
    # It is NOT treated as a semantic document title.
    title_candidate = Path(filename).stem

    extracted_metadata = {
        "title": title_candidate,
        "author": None,
        "created_date": None,
        "language": None,
        "description": None,
        "topics": [],
        "people": [],
        "organizations": [],
        "projects": [],
        "tags": [],
    }

    field_methods = {
        "title": "filename_stem",
        "author": "not_established",
        "created_date": "not_established",
        "language": "not_established",
        "description": "not_established",
        "topics": "not_established",
        "people": "not_established",
        "organizations": "not_established",
        "projects": "not_established",
        "tags": "not_established",
    }

    return {
        "filename": filename,

        "source": {
            "sha256": extracted_artifact["source"]["sha256"],
        },

        "extracted_metadata": extracted_metadata,

        "field_methods": field_methods,

        "content_facts": {
            "character_count": len(source_text),
            "representation_type": representation["type"],
        },

        "provenance": {
            "method": "deterministic_metadata_extraction",
            "source_representation": "extracted_text",
            "ai_used": False,
            "human_confirmed": False,
        },

        "status": "METADATA_EXTRACTED",
    }


def build_prefill_fields(metadata_result: dict):
    """
    Convert extracted metadata into field-level prefill records.

    Values remain unconfirmed until human review.
    """

    fields = {}

    for field_name, value in (
        metadata_result["extracted_metadata"].items()
    ):
        fields[field_name] = {
            "value": value,
            "source": "extracted",
            "method": metadata_result["field_methods"][field_name],
            "user_confirmed": False,
        }

    return fields
'''

metadata_module.write_text(
    metadata_code,
    encoding="utf-8"
)

print("CEREBRO — Metadata Capability")
print("=" * 60)
print(f"Created : {metadata_module}")
print()
print("✓ metadata.py created")

CEREBRO — Metadata Capability
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/metadata.py

✓ metadata.py created


### 8B — Import and run

In [26]:
import metadata
importlib.reload(metadata)

metadata_result = metadata.extract_text_metadata(
    extracted_artifact
)

prefill_fields = metadata.build_prefill_fields(
    metadata_result
)

print("CEREBRO — Deterministic Metadata")
print("=" * 60)

print(json.dumps(
    metadata_result,
    indent=2,
    ensure_ascii=False
))

print("\nCEREBRO — Prefill Fields")
print("=" * 60)

print(json.dumps(
    prefill_fields,
    indent=2,
    ensure_ascii=False
))

CEREBRO — Deterministic Metadata
{
  "filename": "benchmark_001.txt",
  "source": {
    "sha256": "7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832"
  },
  "extracted_metadata": {
    "title": "benchmark_001",
    "author": null,
    "created_date": null,
    "language": null,
    "description": null,
    "topics": [],
    "people": [],
    "organizations": [],
    "projects": [],
    "tags": []
  },
  "field_methods": {
    "title": "filename_stem",
    "author": "not_established",
    "created_date": "not_established",
    "language": "not_established",
    "description": "not_established",
    "topics": "not_established",
    "people": "not_established",
    "organizations": "not_established",
    "projects": "not_established",
    "tags": "not_established"
  },
  "content_facts": {
    "character_count": 294,
    "representation_type": "text"
  },
  "provenance": {
    "method": "deterministic_metadata_extraction",
    "source_representation": "extracted_text",
    

### 8C — Validate the contract

In [27]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Metadata Regression
# ---------------------------------------------------------

assert (
    metadata_result["extracted_metadata"]["title"]
    == "benchmark_001"
)

assert (
    metadata_result["extracted_metadata"]["author"]
    is None
)

assert (
    metadata_result["extracted_metadata"]["created_date"]
    is None
)

assert (
    metadata_result["extracted_metadata"]["topics"]
    == []
)

assert (
    metadata_result["extracted_metadata"]["people"]
    == []
)

assert (
    metadata_result["extracted_metadata"]["projects"]
    == []
)

assert (
    metadata_result["source"]["sha256"]
    == extracted_artifact["source"]["sha256"]
)

assert metadata_result["provenance"]["ai_used"] is False

assert metadata_result["provenance"]["human_confirmed"] is False

assert (
    metadata_result["status"]
    == "METADATA_EXTRACTED"
)

for field_name, field in prefill_fields.items():

    assert field["source"] == "extracted"
    assert field["user_confirmed"] is False


print("CEREBRO — Metadata Regression")
print("=" * 60)

print("✓ Filename-derived title candidate preserved")
print("✓ Unknown author remains unknown")
print("✓ Unknown date remains unknown")
print("✓ No semantic topics manufactured")
print("✓ Source checksum propagated")
print("✓ Field-level provenance preserved")
print("✓ No AI introduced")
print("✓ No human confirmation implied")

print("\n✓ REUSABLE METADATA CAPABILITY PASS")

CEREBRO — Metadata Regression
✓ Filename-derived title candidate preserved
✓ Unknown author remains unknown
✓ Unknown date remains unknown
✓ No semantic topics manufactured
✓ Source checksum propagated
✓ Field-level provenance preserved
✓ No AI introduced
✓ No human confirmation implied

✓ REUSABLE METADATA CAPABILITY PASS


### 9A — Create routing.py

In [28]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Reusable AI Routing Capability
# ---------------------------------------------------------

routing_module = repo_root / "poc/src/routing.py"

routing_code = '''\
"""
CEREBRO AI routing capability.

Determines whether a task should use:
- no AI,
- local AI,
- frontier AI.

The router makes a decision only.
It does NOT invoke a model.

Principle:
Use the lightest capable model.
Escalate only when necessary and permitted.
"""


def route_ai_task(
    task: str,
    *,
    ai_required: bool = True,
    privacy_sensitive: bool = True,
    prefer_local: bool = True,
    local_available: bool = True,
    frontier_allowed: bool = True,
    escalation_allowed: bool = True,
    complexity: str = "low",
    required_quality: str = "standard",
):
    """
    Produce an explainable CEREBRO routing decision.
    """

    valid_complexity = {"low", "medium", "high"}
    valid_quality = {"standard", "high"}

    if complexity not in valid_complexity:
        raise ValueError(
            f"Unsupported complexity: {complexity}"
        )

    if required_quality not in valid_quality:
        raise ValueError(
            f"Unsupported required_quality: {required_quality}"
        )

    factors = {
        "privacy_sensitive": privacy_sensitive,
        "prefer_local": prefer_local,
        "local_available": local_available,
        "frontier_allowed": frontier_allowed,
        "escalation_allowed": escalation_allowed,
        "complexity": complexity,
        "required_quality": required_quality,
    }

    # -----------------------------------------------------
    # No AI required
    # -----------------------------------------------------

    if not ai_required:
        return {
            "task": task,
            "selected_tier": "none",
            "reason": "Task does not require AI.",
            "factors": factors,
            "fallback_tier": None,
            "status": "ROUTED",
        }

    # -----------------------------------------------------
    # Prefer local capability
    # -----------------------------------------------------

    if prefer_local and local_available:
        selected_tier = "local"

        if escalation_allowed and frontier_allowed:
            fallback_tier = "frontier"
        else:
            fallback_tier = None

        reason = (
            "Local AI selected as the lightest available "
            "capability consistent with routing policy."
        )

    # -----------------------------------------------------
    # Local unavailable
    # -----------------------------------------------------

    elif frontier_allowed:
        selected_tier = "frontier"
        fallback_tier = None

        reason = (
            "Local AI unavailable or not preferred; "
            "frontier AI permitted."
        )

    else:
        selected_tier = "unavailable"
        fallback_tier = None

        reason = (
            "No permitted AI capability is currently available."
        )

    return {
        "task": task,
        "selected_tier": selected_tier,
        "reason": reason,
        "factors": factors,
        "fallback_tier": fallback_tier,
        "status": "ROUTED",
    }


def should_escalate(
    validation_result: dict,
    routing_decision: dict
):
    """
    Determine whether a local result should escalate.

    Uses the validation gates established by the CEREBRO
    AI enrichment experiments.
    """

    required_gates = [
        "structured_output_valid",
        "grounding_passed",
        "required_fields_present",
        "entity_preservation_passed",
    ]

    gates_passed = all(
        validation_result.get(gate, False)
        for gate in required_gates
    )

    unsupported_claims = validation_result.get(
        "unsupported_claims_detected",
        False
    )

    quality_passed = (
        gates_passed
        and not unsupported_claims
    )

    escalation_permitted = (
        routing_decision["factors"]["escalation_allowed"]
        and routing_decision["factors"]["frontier_allowed"]
        and routing_decision["fallback_tier"] == "frontier"
    )

    return {
        "quality_passed": quality_passed,
        "escalate": (
            not quality_passed
            and escalation_permitted
        ),
        "status": (
            "PASS"
            if quality_passed
            else "ESCALATE"
            if escalation_permitted
            else "FAIL"
        ),
    }
'''

routing_module.write_text(
    routing_code,
    encoding="utf-8"
)

print("CEREBRO — Routing Capability")
print("=" * 60)
print(f"Created : {routing_module}")
print()
print("✓ routing.py created")

CEREBRO — Routing Capability
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/routing.py

✓ routing.py created


### 9B — Route our metadata enrichment task

In [29]:
import routing
importlib.reload(routing)

routing_decision = routing.route_ai_task(
    task="metadata_enrichment",
    ai_required=True,
    privacy_sensitive=True,
    prefer_local=True,
    local_available=True,
    frontier_allowed=True,
    escalation_allowed=True,
    complexity="low",
    required_quality="standard",
)

print("CEREBRO — Routing Decision")
print("=" * 60)

print(json.dumps(
    routing_decision,
    indent=2,
    ensure_ascii=False
))

CEREBRO — Routing Decision
{
  "task": "metadata_enrichment",
  "selected_tier": "local",
  "reason": "Local AI selected as the lightest available capability consistent with routing policy.",
  "factors": {
    "privacy_sensitive": true,
    "prefer_local": true,
    "local_available": true,
    "frontier_allowed": true,
    "escalation_allowed": true,
    "complexity": "low",
    "required_quality": "standard"
  },
  "fallback_tier": "frontier",
  "status": "ROUTED"
}


### 9C — Validate normal local route

In [30]:
assert routing_decision["selected_tier"] == "local"

assert routing_decision["fallback_tier"] == "frontier"

assert routing_decision["status"] == "ROUTED"

assert (
    routing_decision["factors"]["privacy_sensitive"]
    is True
)

assert (
    routing_decision["factors"]["prefer_local"]
    is True
)

print("CEREBRO — Router Regression")
print("=" * 60)

print("✓ Local-first routing preserved")
print("✓ Privacy factor preserved")
print("✓ Frontier remains fallback only")
print("✓ Escalation permitted")
print("✓ Router invoked no model")

print("\n✓ REUSABLE ROUTING CAPABILITY PASS")

CEREBRO — Router Regression
✓ Local-first routing preserved
✓ Privacy factor preserved
✓ Frontier remains fallback only
✓ Escalation permitted
✓ Router invoked no model

✓ REUSABLE ROUTING CAPABILITY PASS


### 9D — Test our validated escalation behavior

In [31]:
# ---------------------------------------------------------
# Simulated Local AI Validation Failure
# ---------------------------------------------------------

failed_local_validation = {
    "structured_output_valid": True,
    "grounding_passed": False,
    "required_fields_present": True,
    "unsupported_claims_detected": True,
    "entity_preservation_passed": True,
}

escalation_result = routing.should_escalate(
    validation_result=failed_local_validation,
    routing_decision=routing_decision,
)

print("CEREBRO — Escalation Test")
print("=" * 60)

print(json.dumps(
    escalation_result,
    indent=2
))

assert escalation_result["quality_passed"] is False
assert escalation_result["escalate"] is True
assert escalation_result["status"] == "ESCALATE"

print("\n✓ Failed grounding detected")
print("✓ Unsupported claim detected")
print("✓ Frontier escalation requested")
print("✓ No frontier model actually invoked")

print("\n✓ ROUTER ESCALATION CONTRACT PASS")

CEREBRO — Escalation Test
{
  "quality_passed": false,
  "escalate": true,
  "status": "ESCALATE"
}

✓ Failed grounding detected
✓ Unsupported claim detected
✓ Frontier escalation requested
✓ No frontier model actually invoked

✓ ROUTER ESCALATION CONTRACT PASS


### 10A — Create the Ollama adapter

In [32]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Ollama Adapter
# ---------------------------------------------------------

adapters_dir = repo_root / "poc/adapters"
adapters_dir.mkdir(parents=True, exist_ok=True)

ollama_module = adapters_dir / "ollama.py"

ollama_code = '''\
"""
CEREBRO Ollama adapter.

Third-party/model-specific behavior belongs here.
CEREBRO Core must not depend directly on Ollama.
"""

import json
import urllib.request
import urllib.error


DEFAULT_OLLAMA_URL = "http://localhost:11434"
DEFAULT_MODEL = "llama3.2:latest"


def generate_json(
    prompt: str,
    *,
    model: str = DEFAULT_MODEL,
    base_url: str = DEFAULT_OLLAMA_URL,
    temperature: float = 0.0,
    timeout: int = 120,
):
    """
    Execute a local Ollama generation request and return parsed JSON.
    """

    url = f"{base_url}/api/generate"

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": temperature
        },
    }

    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json"
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(
            request,
            timeout=timeout
        ) as response:
            result = json.loads(
                response.read().decode("utf-8")
            )

    except urllib.error.URLError as exc:
        raise RuntimeError(
            f"Ollama request failed: {exc}"
        ) from exc

    raw_response = result.get("response", "")

    try:
        parsed = json.loads(raw_response)

    except json.JSONDecodeError as exc:
        raise ValueError(
            "Ollama returned invalid JSON."
        ) from exc

    return {
        "model": model,
        "provider": "ollama",
        "response": parsed,
    }
'''

ollama_module.write_text(
    ollama_code,
    encoding="utf-8"
)

print("CEREBRO — Model Adapter")
print("=" * 60)
print(f"Created : {ollama_module}")
print()
print("✓ Ollama adapter created")

CEREBRO — Model Adapter
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/adapters/ollama.py

✓ Ollama adapter created


### 10B — Create the CEREBRO enrichment capability

In [33]:
# ---------------------------------------------------------
# CEREBRO 06.5 — AI Metadata Enrichment Capability
# ---------------------------------------------------------

enrichment_module = repo_root / "poc/src/enrichment.py"

enrichment_code = '''\
"""
CEREBRO AI metadata enrichment capability.

AI output is suggestive only.

It does NOT:
- confirm metadata,
- register artifacts,
- create trusted knowledge.
"""

import json


ENRICHMENT_FIELDS = [
    "title",
    "author",
    "created_date",
    "language",
    "description",
    "topics",
    "people",
    "organizations",
    "projects",
    "tags",
]


def build_metadata_enrichment_prompt(
    extracted_artifact: dict,
    metadata_result: dict,
):
    """
    Build a grounded metadata-enrichment request.
    """

    source_text = (
        extracted_artifact["representation"]["content"]
    )

    existing_metadata = (
        metadata_result["extracted_metadata"]
    )

    schema = {
        "title": None,
        "author": None,
        "created_date": None,
        "language": None,
        "description": None,
        "topics": [],
        "people": [],
        "organizations": [],
        "projects": [],
        "tags": [],
    }

    return f"""
You are performing metadata enrichment for CEREBRO.

Use ONLY the supplied artifact content as evidence.

Do not invent information.

If an author, date, person, organization, project or other
fact cannot be established from the artifact, return null
or an empty list.

Return valid JSON only.

Required schema:

{json.dumps(schema, indent=2)}

Existing deterministic metadata:

{json.dumps(existing_metadata, indent=2)}

ARTIFACT CONTENT:

--- BEGIN ARTIFACT ---
{source_text}
--- END ARTIFACT ---
""".strip()


def build_ai_prefill(
    ai_result: dict,
):
    """
    Convert model output into CEREBRO field-level suggestions.

    AI values remain unconfirmed.
    """

    response = ai_result["response"]

    fields = {}

    for field_name in ENRICHMENT_FIELDS:

        value = response.get(field_name)

        fields[field_name] = {
            "value": value,
            "source": "ai_suggested",
            "method": "ai_metadata_enrichment",
            "model": ai_result["model"],
            "provider": ai_result["provider"],
            "status": "suggested",
            "user_confirmed": False,
        }

    return fields
'''

enrichment_module.write_text(
    enrichment_code,
    encoding="utf-8"
)

print("CEREBRO — Enrichment Capability")
print("=" * 60)
print(f"Created : {enrichment_module}")
print()
print("✓ enrichment.py created")

CEREBRO — Enrichment Capability
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/enrichment.py

✓ enrichment.py created


### 10C — Import the capability and adapter

In [34]:
adapters_path = repo_root / "poc/adapters"

if str(adapters_path) not in sys.path:
    sys.path.insert(0, str(adapters_path))

import enrichment
import ollama

importlib.reload(enrichment)
importlib.reload(ollama)

print("✓ CEREBRO enrichment capability imported")
print("✓ Ollama adapter imported")

✓ CEREBRO enrichment capability imported
✓ Ollama adapter imported


### 10D — Build the prompt, but don’t call Ollama yet

In [35]:
enrichment_prompt = (
    enrichment.build_metadata_enrichment_prompt(
        extracted_artifact,
        metadata_result
    )
)

print("CEREBRO — AI Enrichment Request")
print("=" * 60)

print(f"Route : {routing_decision['selected_tier']}")
print(f"Model : {ollama.DEFAULT_MODEL}")
print(f"Characters in prompt : {len(enrichment_prompt):,}")

assert routing_decision["selected_tier"] == "local"

print("\n✓ Local enrichment request prepared")

CEREBRO — AI Enrichment Request
Route : local
Model : llama3.2:latest
Characters in prompt : 1,101

✓ Local enrichment request prepared


### 10E — Execute local enrichment

In [36]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Execute Local AI Enrichment
# ---------------------------------------------------------

assert routing_decision["selected_tier"] == "local"

ai_result = ollama.generate_json(
    enrichment_prompt,
    model="llama3.2:latest",
    temperature=0.0,
)

ai_prefill = enrichment.build_ai_prefill(
    ai_result
)

print("CEREBRO — AI Suggested Metadata")
print("=" * 60)

print(json.dumps(
    ai_prefill,
    indent=2,
    ensure_ascii=False
))

CEREBRO — AI Suggested Metadata
{
  "title": {
    "value": "benchmark_001",
    "source": "ai_suggested",
    "method": "ai_metadata_enrichment",
    "model": "llama3.2:latest",
    "provider": "ollama",
    "status": "suggested",
    "user_confirmed": false
  },
  "author": {
    "value": null,
    "source": "ai_suggested",
    "method": "ai_metadata_enrichment",
    "model": "llama3.2:latest",
    "provider": "ollama",
    "status": "suggested",
    "user_confirmed": false
  },
  "created_date": {
    "value": null,
    "source": "ai_suggested",
    "method": "ai_metadata_enrichment",
    "model": "llama3.2:latest",
    "provider": "ollama",
    "status": "suggested",
    "user_confirmed": false
  },
  "language": {
    "value": null,
    "source": "ai_suggested",
    "method": "ai_metadata_enrichment",
    "model": "llama3.2:latest",
    "provider": "ollama",
    "status": "suggested",
    "user_confirmed": false
  },
  "description": {
    "value": "CEREBRO is a Digital Knowledge 

### 10F — Integrity validation

In [37]:
# ---------------------------------------------------------
# CEREBRO 06.5 — AI Suggestion Integrity
# ---------------------------------------------------------

assert ai_prefill

for field_name, field in ai_prefill.items():

    assert field["source"] == "ai_suggested"

    assert field["status"] == "suggested"

    assert field["user_confirmed"] is False

    assert field["model"] == "llama3.2:latest"

    assert field["provider"] == "ollama"


print("CEREBRO — AI Enrichment Integrity")
print("=" * 60)

print("✓ AI output remains suggestion")
print("✓ AI source explicitly recorded")
print("✓ Model provenance recorded")
print("✓ Provider provenance recorded")
print("✓ No field marked user-confirmed")
print("✓ Artifact not registered")
print("✓ No knowledge fragments created")

print("\n✓ REUSABLE AI ENRICHMENT CAPABILITY PASS")

CEREBRO — AI Enrichment Integrity
✓ AI output remains suggestion
✓ AI source explicitly recorded
✓ Model provenance recorded
✓ Provider provenance recorded
✓ No field marked user-confirmed
✓ Artifact not registered
✓ No knowledge fragments created

✓ REUSABLE AI ENRICHMENT CAPABILITY PASS


### 11A — Add reusable validation to evaluation.py

In [38]:
# ---------------------------------------------------------
# CEREBRO 06.5 — AI Enrichment Evaluation Capability
# ---------------------------------------------------------

evaluation_module = repo_root / "poc/src/evaluation.py"

evaluation_code = '''\
"""
CEREBRO evaluation capabilities.

Evaluates derived outputs before they cross trust boundaries.

This module does not:
- approve AI suggestions,
- register artifacts,
- create trusted knowledge,
- invoke models.
"""


REQUIRED_ENRICHMENT_FIELDS = {
    "title",
    "author",
    "created_date",
    "language",
    "description",
    "topics",
    "people",
    "organizations",
    "projects",
    "tags",
}


def validate_enrichment_structure(ai_prefill: dict):
    """
    Validate structural integrity of CEREBRO AI metadata suggestions.
    """

    if not isinstance(ai_prefill, dict):
        return False

    if not REQUIRED_ENRICHMENT_FIELDS.issubset(
        ai_prefill.keys()
    ):
        return False

    for field_name in REQUIRED_ENRICHMENT_FIELDS:

        field = ai_prefill.get(field_name)

        if not isinstance(field, dict):
            return False

        if field.get("source") != "ai_suggested":
            return False

        if field.get("status") != "suggested":
            return False

        if field.get("user_confirmed") is not False:
            return False

    return True


def validate_required_fields(ai_prefill: dict):
    """
    Required fields means required contract fields are present.

    It does NOT mean every field must contain a value.
    Unknown values are valid.
    """

    return REQUIRED_ENRICHMENT_FIELDS.issubset(
        ai_prefill.keys()
    )


def detect_unsupported_claims(
    ai_prefill: dict,
    source_text: str,
):
    """
    Conservative deterministic grounding check.

    Checks explicit entity-like list values against source text.

    This is intentionally limited and is NOT a complete
    semantic hallucination detector.
    """

    source_lower = source_text.lower()

    evidence_fields = [
        "people",
        "organizations",
        "projects",
    ]

    unsupported = []

    for field_name in evidence_fields:

        field = ai_prefill.get(field_name, {})
        values = field.get("value")

        if values is None:
            continue

        if not isinstance(values, list):
            values = [values]

        for value in values:

            if not isinstance(value, str):
                continue

            candidate = value.strip()

            if (
                candidate
                and candidate.lower() not in source_lower
            ):
                unsupported.append({
                    "field": field_name,
                    "value": candidate,
                })

    return unsupported


def validate_entity_preservation(
    ai_prefill: dict,
    source_text: str,
):
    """
    Validate that suggested explicit entities can be traced
    to source text.

    This is a deterministic PoC gate, not semantic NER scoring.
    """

    unsupported = detect_unsupported_claims(
        ai_prefill,
        source_text
    )

    return len(unsupported) == 0


def evaluate_ai_enrichment(
    ai_prefill: dict,
    source_text: str,
):
    """
    Produce the validation gates expected by the CEREBRO Router.
    """

    structured_output_valid = (
        validate_enrichment_structure(ai_prefill)
    )

    required_fields_present = (
        validate_required_fields(ai_prefill)
    )

    unsupported_claims = (
        detect_unsupported_claims(
            ai_prefill,
            source_text
        )
    )

    unsupported_claims_detected = (
        len(unsupported_claims) > 0
    )

    entity_preservation_passed = (
        validate_entity_preservation(
            ai_prefill,
            source_text
        )
    )

    # Current deterministic grounding gate.
    # Deliberately conservative for the PoC.
    grounding_passed = (
        structured_output_valid
        and not unsupported_claims_detected
    )

    gates = {
        "structured_output_valid":
            structured_output_valid,

        "grounding_passed":
            grounding_passed,

        "required_fields_present":
            required_fields_present,

        "unsupported_claims_detected":
            unsupported_claims_detected,

        "entity_preservation_passed":
            entity_preservation_passed,
    }

    passed = (
        structured_output_valid
        and grounding_passed
        and required_fields_present
        and not unsupported_claims_detected
        and entity_preservation_passed
    )

    return {
        **gates,

        "unsupported_claims":
            unsupported_claims,

        "validation_status":
            "PASS" if passed else "FAIL",
    }
'''

evaluation_module.write_text(
    evaluation_code,
    encoding="utf-8"
)

print("CEREBRO — Evaluation Capability")
print("=" * 60)
print(f"Created : {evaluation_module}")
print()
print("✓ evaluation.py created")

CEREBRO — Evaluation Capability
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/evaluation.py

✓ evaluation.py created


### 11B — Evaluate the actual local AI result

In [39]:
import evaluation
importlib.reload(evaluation)

source_text = (
    extracted_artifact["representation"]["content"]
)

ai_validation = evaluation.evaluate_ai_enrichment(
    ai_prefill=ai_prefill,
    source_text=source_text,
)

print("CEREBRO — AI Enrichment Validation")
print("=" * 60)

print(json.dumps(
    ai_validation,
    indent=2,
    ensure_ascii=False
))

CEREBRO — AI Enrichment Validation
{
  "structured_output_valid": true,
  "grounding_passed": true,
  "required_fields_present": true,
  "unsupported_claims_detected": false,
  "entity_preservation_passed": true,
  "unsupported_claims": [],
  "validation_status": "PASS"
}


### 11C — Feed validation back into Router

In [40]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Quality → Routing Feedback
# ---------------------------------------------------------

routing_feedback = routing.should_escalate(
    validation_result=ai_validation,
    routing_decision=routing_decision,
)

print("CEREBRO — Routing Feedback")
print("=" * 60)

print(json.dumps(
    routing_feedback,
    indent=2
))

CEREBRO — Routing Feedback
{
  "quality_passed": true,
  "escalate": false,
  "status": "PASS"
}


### 11D — Evaluation integrity check

In [41]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Evaluation Contract Validation
# ---------------------------------------------------------

required_gates = {
    "structured_output_valid",
    "grounding_passed",
    "required_fields_present",
    "unsupported_claims_detected",
    "entity_preservation_passed",
}

assert required_gates.issubset(
    ai_validation.keys()
)

assert ai_validation["validation_status"] in {
    "PASS",
    "FAIL",
}

assert routing_feedback["status"] in {
    "PASS",
    "ESCALATE",
    "FAIL",
}

print("CEREBRO — Evaluation Contract")
print("=" * 60)

print("✓ Structured-output gate available")
print("✓ Grounding gate available")
print("✓ Required-fields gate available")
print("✓ Unsupported-claims gate available")
print("✓ Entity-preservation gate available")
print("✓ Validation feeds Router")
print("✓ No automatic human approval")
print("✓ No automatic artifact registration")

print("\n✓ REUSABLE EVALUATION CAPABILITY PASS")

CEREBRO — Evaluation Contract
✓ Structured-output gate available
✓ Grounding gate available
✓ Required-fields gate available
✓ Unsupported-claims gate available
✓ Entity-preservation gate available
✓ Validation feeds Router
✓ No automatic human approval
✓ No automatic artifact registration

✓ REUSABLE EVALUATION CAPABILITY PASS


### 12A — Create review.py

In [42]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Human Review Capability
# ---------------------------------------------------------

review_module = repo_root / "poc/src/review.py"

review_code = '''\
"""
CEREBRO human review capability.

AI-generated metadata remains a suggestion until explicitly
reviewed and approved by a human.

This module does NOT:
- register artifacts,
- create knowledge fragments,
- promote candidate relationships,
- invoke AI models.
"""

from copy import deepcopy
from datetime import datetime, timezone


def prepare_review(ai_prefill: dict):
    """
    Create a reviewable copy of AI-suggested metadata.
    """

    reviewed_prefill = deepcopy(ai_prefill)

    for field_name, field in reviewed_prefill.items():
        field["original_ai_value"] = deepcopy(
            field.get("value")
        )

        field["final_value"] = None
        field["action"] = "pending"
        field["user_confirmed"] = False
        field["status"] = "pending_review"

    return reviewed_prefill


def review_field(
    reviewed_prefill: dict,
    field_name: str,
    final_value,
):
    """
    Apply explicit human review to one metadata field.
    """

    if field_name not in reviewed_prefill:
        raise KeyError(
            f"Unknown review field: {field_name}"
        )

    field = reviewed_prefill[field_name]

    original_value = field.get("original_ai_value")

    if final_value == original_value:
        action = "accepted"
    else:
        action = "edited"

    field["final_value"] = deepcopy(final_value)
    field["action"] = action
    field["user_confirmed"] = True
    field["status"] = "confirmed"

    return reviewed_prefill


def finalize_review(reviewed_prefill: dict):
    """
    Finalize review only when every field has received
    explicit human confirmation.
    """

    unconfirmed = [
        field_name
        for field_name, field in reviewed_prefill.items()
        if field.get("user_confirmed") is not True
    ]

    if unconfirmed:
        raise ValueError(
            "Cannot finalize review. "
            f"Unconfirmed fields: {unconfirmed}"
        )

    final_metadata = {
        field_name: deepcopy(field["final_value"])
        for field_name, field in reviewed_prefill.items()
    }

    return {
        "metadata": final_metadata,

        "review": deepcopy(reviewed_prefill),

        "human_reviewed": True,
        "human_approved": True,

        "reviewed_at": (
            datetime.now(timezone.utc).isoformat()
        ),

        "status": "APPROVED",
    }
'''

review_module.write_text(
    review_code,
    encoding="utf-8"
)

print("CEREBRO — Human Review Capability")
print("=" * 60)
print(f"Created : {review_module}")
print()
print("✓ review.py created")

CEREBRO — Human Review Capability
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/review.py

✓ review.py created


### 12B — Prepare the AI output for review

In [43]:
import review
importlib.reload(review)

reviewed_prefill = review.prepare_review(
    ai_prefill
)

print("CEREBRO — Review Prepared")
print("=" * 60)

for field_name, field in reviewed_prefill.items():
    print(
        f"{field_name:15} "
        f"{field['status']:16} "
        f"{field['original_ai_value']}"
    )

assert all(
    field["user_confirmed"] is False
    for field in reviewed_prefill.values()
)

print("\n✓ AI suggestions remain unconfirmed")

CEREBRO — Review Prepared
title           pending_review   benchmark_001
author          pending_review   None
created_date    pending_review   None
language        pending_review   None
description     pending_review   CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge. It maintains provenance between knowledge fragments and their original source artifacts. The system supports assisted recollection by allowing users to navigate from knowledge back to its supporting evidence.
topics          pending_review   []
people          pending_review   []
organizations   pending_review   []
projects        pending_review   []
tags            pending_review   []

✓ AI suggestions remain unconfirmed


### 12C — Apply our previously validated human decisions

In [44]:
# ---------------------------------------------------------
# Reproduce EXP-INGEST-005 Human Decisions
# ---------------------------------------------------------

human_decisions = {
    "title": "CEREBRO Digital Knowledge Twin",

    "author": None,

    "created_date": None,

    "language": "English",

    "description": (
        "CEREBRO is a Digital Knowledge Twin designed "
        "to preserve and connect human knowledge with "
        "provenance-aware assisted recollection."
    ),

    "topics": [
        "Digital Knowledge Twin",
        "Provenance",
        "Knowledge Fragments",
        "Source Artifacts",
        "Assisted Recollection",
    ],

    "people": [],

    "organizations": [],

    "projects": [
        "CEREBRO"
    ],

    "tags": [
        "CEREBRO",
        "knowledge",
        "provenance",
        "recollection",
    ],
}


for field_name, final_value in human_decisions.items():

    review.review_field(
        reviewed_prefill,
        field_name,
        final_value,
    )


print("CEREBRO — Human Review")
print("=" * 60)

for field_name, field in reviewed_prefill.items():

    print(
        f"{field_name:15} "
        f"{field['action']:10} "
        f"{field['final_value']}"
    )

CEREBRO — Human Review
title           edited     CEREBRO Digital Knowledge Twin
author          accepted   None
created_date    accepted   None
language        edited     English
description     edited     CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge with provenance-aware assisted recollection.
topics          edited     ['Digital Knowledge Twin', 'Provenance', 'Knowledge Fragments', 'Source Artifacts', 'Assisted Recollection']
people          accepted   []
organizations   accepted   []
projects        edited     ['CEREBRO']
tags            edited     ['CEREBRO', 'knowledge', 'provenance', 'recollection']


### 12D — Finalize approval

In [45]:
approved_artifact_metadata = (
    review.finalize_review(
        reviewed_prefill
    )
)

print("CEREBRO — Human Approval")
print("=" * 60)

print(
    json.dumps(
        approved_artifact_metadata["metadata"],
        indent=2,
        ensure_ascii=False
    )
)

print()
print(
    "Human reviewed :",
    approved_artifact_metadata["human_reviewed"]
)

print(
    "Human approved :",
    approved_artifact_metadata["human_approved"]
)

print(
    "Status         :",
    approved_artifact_metadata["status"]
)

CEREBRO — Human Approval
{
  "title": "CEREBRO Digital Knowledge Twin",
  "author": null,
  "created_date": null,
  "language": "English",
  "description": "CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge with provenance-aware assisted recollection.",
  "topics": [
    "Digital Knowledge Twin",
    "Provenance",
    "Knowledge Fragments",
    "Source Artifacts",
    "Assisted Recollection"
  ],
  "people": [],
  "organizations": [],
  "projects": [
    "CEREBRO"
  ],
  "tags": [
    "CEREBRO",
    "knowledge",
    "provenance",
    "recollection"
  ]
}

Human reviewed : True
Human approved : True
Status         : APPROVED


### 12E — Trust-boundary regression

In [46]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Human Approval Regression
# ---------------------------------------------------------

assert (
    approved_artifact_metadata["human_reviewed"]
    is True
)

assert (
    approved_artifact_metadata["human_approved"]
    is True
)

assert (
    approved_artifact_metadata["status"]
    == "APPROVED"
)

assert (
    approved_artifact_metadata["metadata"]["title"]
    == "CEREBRO Digital Knowledge Twin"
)

assert (
    approved_artifact_metadata["metadata"]["author"]
    is None
)

assert (
    approved_artifact_metadata["metadata"]["created_date"]
    is None
)

assert (
    approved_artifact_metadata["metadata"]["people"]
    == []
)

assert (
    approved_artifact_metadata["metadata"]["organizations"]
    == []
)

assert (
    approved_artifact_metadata["metadata"]["projects"]
    == ["CEREBRO"]
)

for field_name, field in (
    approved_artifact_metadata["review"].items()
):
    assert field["user_confirmed"] is True
    assert field["status"] == "confirmed"


print("CEREBRO — Trust Boundary")
print("=" * 60)

print("✓ Every field explicitly reviewed")
print("✓ AI suggestions preserved")
print("✓ Human edits preserved")
print("✓ Unknown author remains unknown")
print("✓ Unknown date remains unknown")
print("✓ Human review recorded")
print("✓ Human approval recorded")

print()
print("AI SUGGESTION")
print("      ↓")
print(" HUMAN REVIEW")
print("      ↓")
print("   APPROVED")
print("      ↓")
print("REGISTRATION ALLOWED")

print("\n✓ REUSABLE HUMAN REVIEW CAPABILITY PASS")

CEREBRO — Trust Boundary
✓ Every field explicitly reviewed
✓ AI suggestions preserved
✓ Human edits preserved
✓ Unknown author remains unknown
✓ Unknown date remains unknown
✓ Human review recorded
✓ Human approval recorded

AI SUGGESTION
      ↓
 HUMAN REVIEW
      ↓
   APPROVED
      ↓
REGISTRATION ALLOWED

✓ REUSABLE HUMAN REVIEW CAPABILITY PASS


### 13 — Reusable Artifact Registration

### 13A — Create registration.py

In [47]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Artifact Registration Capability
# ---------------------------------------------------------

registration_module = repo_root / "poc/src/registration.py"

registration_code = '''\
"""
CEREBRO artifact registration capability.

Registers an approved artifact into the trusted artifact layer.

Registration requires:
- deterministic source inspection,
- source checksum,
- human review,
- explicit human approval.

This module does NOT:
- create knowledge fragments,
- generate embeddings,
- create relationships,
- invoke AI models.
"""

from copy import deepcopy
from datetime import datetime, timezone
import hashlib


def register_artifact(
    artifact_id: str,
    *,
    filename: str,
    content: bytes,
    inspection: dict,
    approved_metadata: dict,
):
    """
    Register a human-approved artifact.

    Fails closed if integrity requirements are not satisfied.
    """

    # -----------------------------------------------------
    # Approval gate
    # -----------------------------------------------------

    if approved_metadata.get("human_reviewed") is not True:
        raise ValueError(
            "Artifact registration rejected: "
            "human_reviewed must be True."
        )

    if approved_metadata.get("human_approved") is not True:
        raise ValueError(
            "Artifact registration rejected: "
            "human_approved must be True."
        )

    if approved_metadata.get("status") != "APPROVED":
        raise ValueError(
            "Artifact registration rejected: "
            "review status must be APPROVED."
        )

    # -----------------------------------------------------
    # Source integrity
    # -----------------------------------------------------

    actual_sha256 = hashlib.sha256(content).hexdigest()

    inspected_sha256 = (
        inspection
        .get("file", {})
        .get("sha256")
    )

    if not inspected_sha256:
        raise ValueError(
            "Artifact registration rejected: "
            "source checksum missing."
        )

    if actual_sha256 != inspected_sha256:
        raise ValueError(
            "Artifact registration rejected: "
            "source checksum mismatch."
        )

    # -----------------------------------------------------
    # Registration
    # -----------------------------------------------------

    registered_at = (
        datetime.now(timezone.utc).isoformat()
    )

    return {
        "artifact_id": artifact_id,

        "filename": filename,

        "metadata": deepcopy(
            approved_metadata["metadata"]
        ),

        "source": {
            "sha256": actual_sha256,
            "size_bytes": len(content),
            "modality": (
                inspection
                .get("identification", {})
                .get("detected_modality")
            ),
        },

        "review": {
            "human_reviewed": True,
            "human_approved": True,
            "reviewed_at": (
                approved_metadata.get("reviewed_at")
            ),
        },

        "provenance": {
            "inspection_method": (
                inspection
                .get("provenance", {})
                .get("method")
            ),

            "source_verified": True,

            "registered_at": registered_at,
        },

        "integrity": {
            "checksum_verified": True,
            "provenance_present": True,
            "human_approval_verified": True,
        },

        "status": "REGISTERED",
    }
'''

registration_module.write_text(
    registration_code,
    encoding="utf-8"
)

print("CEREBRO — Registration Capability")
print("=" * 60)
print(f"Created : {registration_module}")
print()
print("✓ registration.py created")

CEREBRO — Registration Capability
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/registration.py

✓ registration.py created


### 13B — Register the benchmark artifact

In [48]:
import registration
importlib.reload(registration)

registered_artifact = registration.register_artifact(
    artifact_id="ART-0001",
    filename=filename,
    content=file_bytes,
    inspection=reusable_inspection,
    approved_metadata=approved_artifact_metadata,
)

print("CEREBRO — Registered Artifact")
print("=" * 60)

print(json.dumps(
    registered_artifact,
    indent=2,
    ensure_ascii=False
))

CEREBRO — Registered Artifact
{
  "artifact_id": "ART-0001",
  "filename": "benchmark_001.txt",
  "metadata": {
    "title": "CEREBRO Digital Knowledge Twin",
    "author": null,
    "created_date": null,
    "language": "English",
    "description": "CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge with provenance-aware assisted recollection.",
    "topics": [
      "Digital Knowledge Twin",
      "Provenance",
      "Knowledge Fragments",
      "Source Artifacts",
      "Assisted Recollection"
    ],
    "people": [],
    "organizations": [],
    "projects": [
      "CEREBRO"
    ],
    "tags": [
      "CEREBRO",
      "knowledge",
      "provenance",
      "recollection"
    ]
  },
  "source": {
    "sha256": "7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832",
    "size_bytes": 294,
    "modality": "text"
  },
  "review": {
    "human_reviewed": true,
    "human_approved": true,
    "reviewed_at": "2026-09-24T08:01:49.653438+00:00"

### 13C — CIF registration regression

In [49]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Registration Integrity Regression
# ---------------------------------------------------------

assert registered_artifact["artifact_id"] == "ART-0001"

assert registered_artifact["status"] == "REGISTERED"

assert (
    registered_artifact["source"]["sha256"]
    == reusable_inspection["file"]["sha256"]
)

assert (
    registered_artifact["provenance"]["source_verified"]
    is True
)

assert (
    registered_artifact["review"]["human_reviewed"]
    is True
)

assert (
    registered_artifact["review"]["human_approved"]
    is True
)

assert (
    registered_artifact["integrity"]["checksum_verified"]
    is True
)

assert (
    registered_artifact["integrity"]["provenance_present"]
    is True
)

assert (
    registered_artifact["integrity"]["human_approval_verified"]
    is True
)

print("CEREBRO — Registration Integrity")
print("=" * 60)

print("✓ ART-0001 identity preserved")
print("✓ Source checksum verified")
print("✓ Source provenance retained")
print("✓ Human review verified")
print("✓ Human approval verified")
print("✓ Artifact registered")
print("✓ No knowledge fragments created")
print("✓ No embeddings generated")
print("✓ No relationships generated")

print("\n✓ REUSABLE REGISTRATION CAPABILITY PASS")

CEREBRO — Registration Integrity
✓ ART-0001 identity preserved
✓ Source checksum verified
✓ Source provenance retained
✓ Human review verified
✓ Human approval verified
✓ Artifact registered
✓ No knowledge fragments created
✓ No embeddings generated
✓ No relationships generated

✓ REUSABLE REGISTRATION CAPABILITY PASS


### 13D — Test the guardrail itself

In [50]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Negative Registration Test
# ---------------------------------------------------------

from copy import deepcopy

unapproved_metadata = deepcopy(
    approved_artifact_metadata
)

unapproved_metadata["human_approved"] = False
unapproved_metadata["status"] = "PENDING_REVIEW"

registration_blocked = False

try:

    registration.register_artifact(
        artifact_id="ART-TEST-REJECT",
        filename=filename,
        content=file_bytes,
        inspection=reusable_inspection,
        approved_metadata=unapproved_metadata,
    )

except ValueError as exc:

    registration_blocked = True

    print("Registration correctly rejected:")
    print(exc)


assert registration_blocked is True

print()
print("✓ Unapproved artifact cannot be registered")
print("✓ FAIL-CLOSED REGISTRATION GUARDRAIL PASS")

Registration correctly rejected:
Artifact registration rejected: human_approved must be True.

✓ Unapproved artifact cannot be registered
✓ FAIL-CLOSED REGISTRATION GUARDRAIL PASS


### 14 — Reusable Knowledge Construction

### 14A — Create chunking.py

In [51]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Reusable Chunking Capability
# ---------------------------------------------------------

chunking_module = repo_root / "poc/src/chunking.py"

chunking_code = '''\
"""
CEREBRO deterministic text segmentation.

Creates source-addressable segments while preserving exact
character offsets into the original artifact.

No AI.
No embeddings.
No semantic enrichment.
"""

import re


def segment_sentences(source_text: str):
    """
    Deterministically segment text into sentences while retaining
    exact source character offsets.
    """

    segments = []

    pattern = re.compile(
        r'[^.!?]+(?:[.!?]+|$)',
        re.MULTILINE
    )

    for match in pattern.finditer(source_text):

        raw_text = match.group(0)

        leading = len(raw_text) - len(raw_text.lstrip())
        trailing_text = raw_text.strip()

        if not trailing_text:
            continue

        start_char = match.start() + leading
        end_char = start_char + len(trailing_text)

        segments.append({
            "text": trailing_text,
            "source_location": {
                "type": "character_range",
                "start_char": start_char,
                "end_char": end_char,
            },
        })

    return segments
'''

chunking_module.write_text(
    chunking_code,
    encoding="utf-8"
)

print("CEREBRO — Chunking Capability")
print("=" * 60)
print(f"Created : {chunking_module}")
print()
print("✓ chunking.py created")

CEREBRO — Chunking Capability
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/chunking.py

✓ chunking.py created


### 14B — Run segmentation

In [52]:
import chunking
importlib.reload(chunking)

source_text = (
    extracted_artifact["representation"]["content"]
)

segments = chunking.segment_sentences(
    source_text
)

print("CEREBRO — Source Segmentation")
print("=" * 60)

for i, segment in enumerate(segments, start=1):

    location = segment["source_location"]

    print(
        f"{i}: "
        f"[{location['start_char']}:{location['end_char']}] "
        f"{segment['text']}"
    )

CEREBRO — Source Segmentation
1: [0:85] CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge.
2: [86:174] It maintains provenance between knowledge fragments and their original source artifacts.
3: [175:294] The system supports assisted recollection by allowing users to navigate from knowledge back to its supporting evidence.


### 14C — Prove exact source reconstruction

In [53]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Source Location Regression
# ---------------------------------------------------------

assert len(segments) == 3

for segment in segments:

    location = segment["source_location"]

    reconstructed = source_text[
        location["start_char"]:
        location["end_char"]
    ]

    assert reconstructed == segment["text"]


print("CEREBRO — Source Addressability")
print("=" * 60)

print(f"✓ Segments          : {len(segments)}")
print("✓ Character offsets : exact")
print("✓ Source evidence   : reconstructable")
print("✓ No AI used")

print("\n✓ REUSABLE CHUNKING CAPABILITY PASS")

CEREBRO — Source Addressability
✓ Segments          : 3
✓ Character offsets : exact
✓ Source evidence   : reconstructable
✓ No AI used

✓ REUSABLE CHUNKING CAPABILITY PASS


### 14D — Create knowledge.py

In [54]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Knowledge Construction Capability
# ---------------------------------------------------------

knowledge_module = repo_root / "poc/src/knowledge.py"

knowledge_code = '''\
"""
CEREBRO knowledge construction.

Constructs provenance-aware Knowledge Fragments from an
approved and registered artifact.

Semantic annotations supplied to this module are controlled
inputs. This module does not infer them automatically.
"""

from copy import deepcopy
from datetime import datetime, timezone


def construct_knowledge_model(
    *,
    knowledge_model_id: str,
    registered_artifact: dict,
    segments: list,
    fragment_annotations: list,
    trusted_relationships: list,
):
    """
    Construct trusted knowledge fragments from controlled,
    validated inputs.
    """

    if registered_artifact.get("status") != "REGISTERED":
        raise ValueError(
            "Knowledge construction requires "
            "a REGISTERED artifact."
        )

    if len(segments) != len(fragment_annotations):
        raise ValueError(
            "Each source segment requires exactly "
            "one controlled fragment annotation."
        )

    artifact_id = registered_artifact["artifact_id"]

    fragments = []

    for segment, annotation in zip(
        segments,
        fragment_annotations
    ):

        fragment = {
            "fragment_id": annotation["fragment_id"],

            "artifact_id": artifact_id,

            "content": segment["text"],

            "source_location": deepcopy(
                segment["source_location"]
            ),

            "concepts": deepcopy(
                annotation.get("concepts", [])
            ),

            "entities": deepcopy(
                annotation.get("entities", [])
            ),

            "temporal": {
                "source_date": annotation.get(
                    "source_date"
                ),

                "knowledge_date": annotation.get(
                    "knowledge_date"
                ),

                "ingested_at": annotation.get(
                    "ingested_at"
                ),
            },

            "provenance": {
                "source_artifact_id": artifact_id,
                "source_sha256":
                    registered_artifact["source"]["sha256"],
                "source_location_preserved": True,
            },

            "status": "TRUSTED",
        }

        fragments.append(fragment)

    return {
        "knowledge_model_id": knowledge_model_id,

        "artifact_id": artifact_id,

        "fragments": fragments,

        "trusted_relationships": deepcopy(
            trusted_relationships
        ),

        "constructed_at": (
            datetime.now(timezone.utc).isoformat()
        ),

        "status": "CONSTRUCTED",
    }
'''

knowledge_module.write_text(
    knowledge_code,
    encoding="utf-8"
)

print("CEREBRO — Knowledge Capability")
print("=" * 60)
print(f"Created : {knowledge_module}")
print()
print("✓ knowledge.py created")

CEREBRO — Knowledge Capability
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/knowledge.py

✓ knowledge.py created


### 14E — Reproduce the validated Step 03 annotations

In [55]:
import knowledge
importlib.reload(knowledge)

ingested_at = (
    registered_artifact["provenance"]["registered_at"]
)

fragment_annotations = [
    {
        "fragment_id": "KF-0001",

        "concepts": [
            "CEREBRO",
            "Digital Knowledge Twin",
            "Human Knowledge",
        ],

        "entities": [],

        "source_date": None,
        "knowledge_date": None,
        "ingested_at": ingested_at,
    },

    {
        "fragment_id": "KF-0002",

        "concepts": [
            "Provenance",
            "Knowledge Fragment",
            "Source Artifact",
        ],

        "entities": [],

        "source_date": None,
        "knowledge_date": None,
        "ingested_at": ingested_at,
    },

    {
        "fragment_id": "KF-0003",

        "concepts": [
            "Assisted Recollection",
            "Knowledge Navigation",
            "Supporting Evidence",
        ],

        "entities": [],

        "source_date": None,
        "knowledge_date": None,
        "ingested_at": ingested_at,
    },
]

### 14F — Construct the Knowledge Model

In [57]:
knowledge_model = knowledge.construct_knowledge_model(
    knowledge_model_id="KM-0001",
    registered_artifact=registered_artifact,
    segments=segments,
    fragment_annotations=fragment_annotations,
    trusted_relationships=trusted_relationships,
)

print("CEREBRO — Knowledge Model")
print("=" * 60)

print(f"Model     : {knowledge_model['knowledge_model_id']}")
print(f"Artifact  : {knowledge_model['artifact_id']}")
print(f"Fragments : {len(knowledge_model['fragments'])}")
print(
    f"Relations : "
    f"{len(knowledge_model['trusted_relationships'])}"
)

print()

for fragment in knowledge_model["fragments"]:

    print(
        fragment["fragment_id"],
        "→",
        fragment["content"]
    )

CEREBRO — Knowledge Model
Model     : KM-0001
Artifact  : ART-0001
Fragments : 3
Relations : 3

KF-0001 → CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge.
KF-0002 → It maintains provenance between knowledge fragments and their original source artifacts.
KF-0003 → The system supports assisted recollection by allowing users to navigate from knowledge back to its supporting evidence.


### 14G — CIF-02 test

In [58]:
# ---------------------------------------------------------
# CEREBRO 06.5 — CIF-02 Regression
#
# Every trusted knowledge fragment must resolve to its
# original artifact and precise source evidence.
# ---------------------------------------------------------

assert knowledge_model["artifact_id"] == "ART-0001"

assert len(knowledge_model["fragments"]) == 3


for fragment in knowledge_model["fragments"]:

    assert fragment["status"] == "TRUSTED"

    assert (
        fragment["artifact_id"]
        == registered_artifact["artifact_id"]
    )

    assert (
        fragment["provenance"]["source_artifact_id"]
        == registered_artifact["artifact_id"]
    )

    assert (
        fragment["provenance"]["source_sha256"]
        == registered_artifact["source"]["sha256"]
    )

    location = fragment["source_location"]

    reconstructed_evidence = source_text[
        location["start_char"]:
        location["end_char"]
    ]

    assert (
        reconstructed_evidence
        == fragment["content"]
    )


print("CEREBRO — CIF-02")
print("=" * 60)

print("✓ KF-0001 → ART-0001 → exact evidence")
print("✓ KF-0002 → ART-0001 → exact evidence")
print("✓ KF-0003 → ART-0001 → exact evidence")

print()
print("✓ Artifact provenance preserved")
print("✓ Source checksum preserved")
print("✓ Exact source locations preserved")
print("✓ Unknown dates remain unknown")
print("✓ Controlled relationships remain trusted")
print("✓ No inferred relationship promoted to trusted")

print("\n✓ CIF-02 KNOWLEDGE LINEAGE PASS")

CEREBRO — CIF-02
✓ KF-0001 → ART-0001 → exact evidence
✓ KF-0002 → ART-0001 → exact evidence
✓ KF-0003 → ART-0001 → exact evidence

✓ Artifact provenance preserved
✓ Source checksum preserved
✓ Exact source locations preserved
✓ Unknown dates remain unknown
✓ Controlled relationships remain trusted
✓ No inferred relationship promoted to trusted

✓ CIF-02 KNOWLEDGE LINEAGE PASS


### 15A — Add embedding support to the Ollama adapter

In [59]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Extend Ollama Adapter: Embeddings
# ---------------------------------------------------------

ollama_module = repo_root / "poc/adapters/ollama.py"

embedding_adapter_code = '''


def generate_embedding(
    text: str,
    *,
    model: str = "mxbai-embed-large:latest",
    base_url: str = DEFAULT_OLLAMA_URL,
    timeout: int = 120,
):
    """
    Generate one local embedding using Ollama.

    Uses the /api/embeddings endpoint validated by
    EXP-KNOW-002.
    """

    url = f"{base_url}/api/embeddings"

    payload = {
        "model": model,
        "prompt": text,
    }

    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json"
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(
            request,
            timeout=timeout
        ) as response:

            result = json.loads(
                response.read().decode("utf-8")
            )

    except urllib.error.URLError as exc:
        raise RuntimeError(
            f"Ollama embedding request failed: {exc}"
        ) from exc

    embedding = result.get("embedding")

    if not isinstance(embedding, list):
        raise ValueError(
            "Ollama returned no valid embedding."
        )

    return {
        "model": model,
        "provider": "ollama",
        "embedding": embedding,
        "dimensions": len(embedding),
    }
'''

current_code = ollama_module.read_text(
    encoding="utf-8"
)

if "def generate_embedding(" not in current_code:

    with ollama_module.open(
        "a",
        encoding="utf-8"
    ) as f:
        f.write(embedding_adapter_code)

    print("✓ generate_embedding() added")

else:
    print("✓ generate_embedding() already present")

print(f"Adapter : {ollama_module}")

✓ generate_embedding() added
Adapter : /Users/joeldizon/development/cerebro_dev/cerebro/poc/adapters/ollama.py


### 15B — Create CEREBRO-owned embedding.py

In [60]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Reusable Embedding Capability
# ---------------------------------------------------------

embedding_module = repo_root / "poc/src/embedding.py"

embedding_code = '''\
"""
CEREBRO embedding capability.

Defines CEREBRO-owned embedding records.

Model execution is performed by an adapter supplied
to this capability.
"""


def embed_fragments(
    fragments: list,
    embedding_function,
):
    """
    Generate embeddings for Knowledge Fragments.

    embedding_function must accept text and return:
    model, provider, embedding, dimensions.
    """

    records = []

    for fragment in fragments:

        result = embedding_function(
            fragment["content"]
        )

        records.append({
            "fragment_id": fragment["fragment_id"],

            "artifact_id": fragment["artifact_id"],

            "embedding": result["embedding"],

            "embedding_metadata": {
                "model": result["model"],
                "provider": result["provider"],
                "dimensions": result["dimensions"],
            },

            "provenance": {
                "source_fragment_id":
                    fragment["fragment_id"],

                "source_artifact_id":
                    fragment["artifact_id"],

                "source_sha256":
                    fragment["provenance"]["source_sha256"],
            },

            "status": "DERIVED",
        })

    return records
'''

embedding_module.write_text(
    embedding_code,
    encoding="utf-8"
)

print("CEREBRO — Embedding Capability")
print("=" * 60)
print(f"Created : {embedding_module}")
print()
print("✓ embedding.py created")

CEREBRO — Embedding Capability
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/embedding.py

✓ embedding.py created


### 15C — Generate the real embeddings

In [61]:
import embedding

importlib.reload(ollama)
importlib.reload(embedding)

embedding_records = embedding.embed_fragments(
    fragments=knowledge_model["fragments"],
    embedding_function=ollama.generate_embedding,
)

print("CEREBRO — Fragment Embeddings")
print("=" * 60)

for record in embedding_records:

    print(
        f"{record['fragment_id']:8} "
        f"{record['embedding_metadata']['model']:28} "
        f"{record['embedding_metadata']['dimensions']} dimensions"
    )

CEREBRO — Fragment Embeddings
KF-0001  mxbai-embed-large:latest     1024 dimensions
KF-0002  mxbai-embed-large:latest     1024 dimensions
KF-0003  mxbai-embed-large:latest     1024 dimensions


### 15D — Embedding regression

In [62]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Embedding Regression
# ---------------------------------------------------------

assert len(embedding_records) == 3

for record in embedding_records:

    assert (
        record["embedding_metadata"]["model"]
        == "mxbai-embed-large:latest"
    )

    assert (
        record["embedding_metadata"]["provider"]
        == "ollama"
    )

    assert (
        record["embedding_metadata"]["dimensions"]
        == 1024
    )

    assert len(record["embedding"]) == 1024

    assert record["status"] == "DERIVED"

    assert record["provenance"]["source_fragment_id"]

    assert (
        record["provenance"]["source_artifact_id"]
        == "ART-0001"
    )


print("CEREBRO — Embedding Regression")
print("=" * 60)

print("✓ 3 knowledge fragments embedded")
print("✓ Model preserved: mxbai-embed-large:latest")
print("✓ Dimensions preserved: 1024")
print("✓ Fragment provenance preserved")
print("✓ Artifact provenance preserved")
print("✓ Embeddings remain derived representations")

print("\n✓ REUSABLE EMBEDDING CAPABILITY PASS")

CEREBRO — Embedding Regression
✓ 3 knowledge fragments embedded
✓ Model preserved: mxbai-embed-large:latest
✓ Dimensions preserved: 1024
✓ Fragment provenance preserved
✓ Artifact provenance preserved
✓ Embeddings remain derived representations

✓ REUSABLE EMBEDDING CAPABILITY PASS


### 15E — Create relationships.py

In [63]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Semantic Candidate Relationships
# ---------------------------------------------------------

relationships_module = (
    repo_root / "poc/src/relationships.py"
)

relationships_code = '''\
"""
CEREBRO relationship discovery.

Embedding similarity may generate candidate relationships.

Candidate relationships MUST NOT be silently promoted
to trusted relationships.
"""

import math


def cosine_similarity(vector_a, vector_b):

    if len(vector_a) != len(vector_b):
        raise ValueError(
            "Embedding dimensions do not match."
        )

    dot = sum(
        a * b
        for a, b in zip(vector_a, vector_b)
    )

    norm_a = math.sqrt(
        sum(a * a for a in vector_a)
    )

    norm_b = math.sqrt(
        sum(b * b for b in vector_b)
    )

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot / (norm_a * norm_b)


def relevance_label(
    similarity: float,
    *,
    high_threshold: float = 0.75,
    related_threshold: float = 0.55,
):
    """
    Thresholds are experimental PoC values inherited
    from EXP-KNOW-002. They are NOT calibrated production
    thresholds.
    """

    if similarity >= high_threshold:
        return "Highly Related"

    if similarity >= related_threshold:
        return "Related"

    return "Potential Connection"


def discover_candidate_relationships(
    embedding_records: list,
    *,
    high_threshold: float = 0.75,
    related_threshold: float = 0.55,
):
    """
    Compare every unique fragment pair and create
    semantic relationship candidates.
    """

    candidates = []

    relationship_number = 1

    for i in range(len(embedding_records)):

        for j in range(
            i + 1,
            len(embedding_records)
        ):

            source = embedding_records[i]
            target = embedding_records[j]

            similarity = cosine_similarity(
                source["embedding"],
                target["embedding"],
            )

            candidates.append({
                "candidate_relationship_id":
                    f"CAND-{relationship_number:04d}",

                "source_fragment_id":
                    source["fragment_id"],

                "target_fragment_id":
                    target["fragment_id"],

                "relationship_type":
                    "SEMANTIC_SIMILARITY",

                "similarity": similarity,

                "relevance": relevance_label(
                    similarity,
                    high_threshold=high_threshold,
                    related_threshold=related_threshold,
                ),

                "method": "cosine_similarity",

                "embedding_model":
                    source[
                        "embedding_metadata"
                    ]["model"],

                "status": "candidate",

                "human_validated": False,
            })

            relationship_number += 1

    return candidates
'''

relationships_module.write_text(
    relationships_code,
    encoding="utf-8"
)

print("CEREBRO — Relationship Capability")
print("=" * 60)
print(f"Created : {relationships_module}")
print()
print("✓ relationships.py created")

CEREBRO — Relationship Capability
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/relationships.py

✓ relationships.py created


### 15F — Discover semantic candidates

In [64]:
import relationships
importlib.reload(relationships)

candidate_relationships = (
    relationships.discover_candidate_relationships(
        embedding_records
    )
)

print("CEREBRO — Semantic Candidates")
print("=" * 72)

for candidate in candidate_relationships:

    print(
        f"{candidate['source_fragment_id']} ↔ "
        f"{candidate['target_fragment_id']} | "
        f"{candidate['similarity']:.4f} | "
        f"{candidate['relevance']} | "
        f"{candidate['status']}"
    )

CEREBRO — Semantic Candidates
KF-0001 ↔ KF-0002 | 0.6841 | Related | candidate
KF-0001 ↔ KF-0003 | 0.7228 | Related | candidate
KF-0002 ↔ KF-0003 | 0.6948 | Related | candidate


### 15G — Trust-boundary regression

In [65]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Candidate Relationship Integrity
# ---------------------------------------------------------

assert len(candidate_relationships) == 3

trusted_ids = {
    relationship["relationship_id"]
    for relationship
    in knowledge_model["trusted_relationships"]
}

for candidate in candidate_relationships:

    assert candidate["status"] == "candidate"

    assert candidate["human_validated"] is False

    assert (
        candidate["relationship_type"]
        == "SEMANTIC_SIMILARITY"
    )

    assert candidate["method"] == "cosine_similarity"

    assert (
        candidate["embedding_model"]
        == "mxbai-embed-large:latest"
    )

    assert (
        candidate["candidate_relationship_id"]
        not in trusted_ids
    )

    assert 0.0 <= candidate["similarity"] <= 1.0


# Existing trusted relationships remain untouched.
assert len(
    knowledge_model["trusted_relationships"]
) == 3

assert all(
    relationship["status"] == "trusted"
    for relationship
    in knowledge_model["trusted_relationships"]
)


print("CEREBRO — Relationship Trust Boundary")
print("=" * 60)

print("✓ 3 semantic comparisons generated")
print("✓ Similarity retained as measured evidence")
print("✓ Human-readable relevance retained")
print("✓ Thresholds remain experimental")
print("✓ All discovered relationships remain candidate")
print("✓ No candidate marked human-validated")
print("✓ Trusted relationships remain untouched")
print("✓ Candidate IDs separated from trusted REL-* IDs")

print()
print("EMBEDDING")
print("    ↓")
print("COSINE SIMILARITY")
print("    ↓")
print("CANDIDATE RELATIONSHIP")
print("    ╳")
print("NO AUTOMATIC PROMOTION TO TRUSTED")

print("\n✓ SEMANTIC RELATIONSHIP GUARDRAIL PASS")

CEREBRO — Relationship Trust Boundary
✓ 3 semantic comparisons generated
✓ Similarity retained as measured evidence
✓ Human-readable relevance retained
✓ Thresholds remain experimental
✓ All discovered relationships remain candidate
✓ No candidate marked human-validated
✓ Trusted relationships remain untouched
✓ Candidate IDs separated from trusted REL-* IDs

EMBEDDING
    ↓
COSINE SIMILARITY
    ↓
CANDIDATE RELATIONSHIP
    ╳
NO AUTOMATIC PROMOTION TO TRUSTED

✓ SEMANTIC RELATIONSHIP GUARDRAIL PASS


### 16 — Reusable Assisted Recollection

### 16A — Create recollection.py

In [66]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Assisted Recollection Capability
# ---------------------------------------------------------

recollection_module = (
    repo_root / "poc/src/recollection.py"
)

recollection_code = '''\
"""
CEREBRO Assisted Recollection.

Deterministically resolves a Knowledge Fragment back to:
- its registered artifact,
- precise source location,
- exact source evidence,
- trusted relationships,
- candidate semantic relationships.

No LLM is required for source resolution.
"""


def find_fragment(
    fragment_id: str,
    knowledge_model: dict,
):
    """
    Resolve a Knowledge Fragment by canonical KF-* identity.
    """

    for fragment in knowledge_model["fragments"]:

        if fragment["fragment_id"] == fragment_id:
            return fragment

    raise KeyError(
        f"Knowledge Fragment not found: {fragment_id}"
    )


def resolve_source_evidence(
    fragment: dict,
    registered_artifact: dict,
    source_text: str,
):
    """
    Resolve exact evidence from the original source representation.
    """

    if (
        fragment["artifact_id"]
        != registered_artifact["artifact_id"]
    ):
        raise ValueError(
            "Fragment artifact does not match "
            "registered artifact."
        )

    if (
        fragment["provenance"]["source_sha256"]
        != registered_artifact["source"]["sha256"]
    ):
        raise ValueError(
            "Fragment source checksum does not match "
            "registered artifact."
        )

    location = fragment["source_location"]

    if location["type"] != "character_range":
        raise ValueError(
            "Unsupported source location type."
        )

    start_char = location["start_char"]
    end_char = location["end_char"]

    evidence = source_text[
        start_char:end_char
    ]

    if evidence != fragment["content"]:
        raise ValueError(
            "Resolved evidence does not match "
            "Knowledge Fragment content."
        )

    return evidence


def find_trusted_relationships(
    fragment_id: str,
    knowledge_model: dict,
):
    """
    Return trusted relationships involving the fragment.
    """

    relationships = []

    for relationship in (
        knowledge_model["trusted_relationships"]
    ):

        if (
            relationship["source_fragment_id"]
            == fragment_id
            or
            relationship["target_fragment_id"]
            == fragment_id
        ):
            relationships.append(relationship)

    return relationships


def find_candidate_relationships(
    fragment_id: str,
    candidate_relationships: list,
):
    """
    Return semantic candidates involving the fragment.

    Candidates remain clearly separated from trusted edges.
    """

    relationships = []

    for relationship in candidate_relationships:

        if (
            relationship["source_fragment_id"]
            == fragment_id
            or
            relationship["target_fragment_id"]
            == fragment_id
        ):
            relationships.append(relationship)

    return relationships


def recollect(
    fragment_id: str,
    *,
    knowledge_model: dict,
    registered_artifact: dict,
    source_text: str,
    candidate_relationships: list = None,
):
    """
    Build a deterministic Assisted Recollection record.
    """

    if candidate_relationships is None:
        candidate_relationships = []

    fragment = find_fragment(
        fragment_id,
        knowledge_model,
    )

    evidence = resolve_source_evidence(
        fragment,
        registered_artifact,
        source_text,
    )

    trusted = find_trusted_relationships(
        fragment_id,
        knowledge_model,
    )

    candidates = find_candidate_relationships(
        fragment_id,
        candidate_relationships,
    )

    return {
        "fragment_id": fragment_id,

        "knowledge": fragment["content"],

        "concepts": fragment.get(
            "concepts",
            []
        ),

        "source": {
            "artifact_id":
                registered_artifact["artifact_id"],

            "filename":
                registered_artifact["filename"],

            "sha256":
                registered_artifact["source"]["sha256"],

            "location":
                fragment["source_location"],

            "evidence":
                evidence,
        },

        "relationships": {
            "trusted": trusted,
            "candidate": candidates,
        },

        "integrity": {
            "source_resolved": True,
            "checksum_verified": True,
            "evidence_verified": True,
        },

        "status": "RECOLLECTED",
    }
'''

recollection_module.write_text(
    recollection_code,
    encoding="utf-8"
)

print("CEREBRO — Assisted Recollection")
print("=" * 60)
print(f"Created : {recollection_module}")
print()
print("✓ recollection.py created")

CEREBRO — Assisted Recollection
Created : /Users/joeldizon/development/cerebro_dev/cerebro/poc/src/recollection.py

✓ recollection.py created


### 16B — Recollect KF-0002

In [67]:
import recollection
importlib.reload(recollection)

recollection_result = recollection.recollect(
    "KF-0002",
    knowledge_model=knowledge_model,
    registered_artifact=registered_artifact,
    source_text=source_text,
    candidate_relationships=candidate_relationships,
)

print("CEREBRO — Assisted Recollection")
print("=" * 60)

print(
    json.dumps(
        recollection_result,
        indent=2,
        ensure_ascii=False
    )
)

CEREBRO — Assisted Recollection
{
  "fragment_id": "KF-0002",
  "knowledge": "It maintains provenance between knowledge fragments and their original source artifacts.",
  "concepts": [
    "Provenance",
    "Knowledge Fragment",
    "Source Artifact"
  ],
  "source": {
    "artifact_id": "ART-0001",
    "filename": "benchmark_001.txt",
    "sha256": "7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832",
    "location": {
      "type": "character_range",
      "start_char": 86,
      "end_char": 174
    },
    "evidence": "It maintains provenance between knowledge fragments and their original source artifacts."
  },
  "relationships": {
    "trusted": [
      {
        "relationship_id": "REL-0001",
        "source_fragment_id": "KF-0001",
        "target_fragment_id": "KF-0002",
        "relationship_type": "SUPPORTED_BY",
        "status": "trusted"
      },
      {
        "relationship_id": "REL-0002",
        "source_fragment_id": "KF-0002",
        "target_fragment_id

### 16C — Human-readable recall cue

In [68]:
result = recollection_result
source = result["source"]

print("CEREBRO — Recall Cue")
print("=" * 60)

print(f"Knowledge : {result['knowledge']}")
print()

print("Concepts:")
for concept in result["concepts"]:
    print(f"  • {concept}")

print()
print(f"Source    : {source['filename']}")
print(f"Artifact  : {source['artifact_id']}")

location = source["location"]

print(
    "Location  : "
    f"characters "
    f"{location['start_char']}"
    f"–{location['end_char']}"
)

print()
print("Exact Evidence:")
print(f'  "{source["evidence"]}"')

print()
print(
    "Trusted relationships :",
    len(
        result["relationships"]["trusted"]
    )
)

print(
    "Candidate relationships:",
    len(
        result["relationships"]["candidate"]
    )
)

CEREBRO — Recall Cue
Knowledge : It maintains provenance between knowledge fragments and their original source artifacts.

Concepts:
  • Provenance
  • Knowledge Fragment
  • Source Artifact

Source    : benchmark_001.txt
Artifact  : ART-0001
Location  : characters 86–174

Exact Evidence:
  "It maintains provenance between knowledge fragments and their original source artifacts."

Trusted relationships : 2
Candidate relationships: 2


### 16D — CIF-01 / CIF-02 regression across ALL fragments

In [69]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Full Recollection Integrity Regression
# ---------------------------------------------------------

for fragment in knowledge_model["fragments"]:

    fragment_id = fragment["fragment_id"]

    result = recollection.recollect(
        fragment_id,
        knowledge_model=knowledge_model,
        registered_artifact=registered_artifact,
        source_text=source_text,
        candidate_relationships=candidate_relationships,
    )

    assert result["fragment_id"] == fragment_id

    assert (
        result["source"]["artifact_id"]
        == "ART-0001"
    )

    assert (
        result["source"]["sha256"]
        == registered_artifact["source"]["sha256"]
    )

    assert (
        result["source"]["evidence"]
        == fragment["content"]
    )

    assert (
        result["integrity"]["source_resolved"]
        is True
    )

    assert (
        result["integrity"]["checksum_verified"]
        is True
    )

    assert (
        result["integrity"]["evidence_verified"]
        is True
    )

    assert result["status"] == "RECOLLECTED"


print("CEREBRO — Recollection Integrity")
print("=" * 60)

print("✓ KF-0001 → exact evidence")
print("✓ KF-0002 → exact evidence")
print("✓ KF-0003 → exact evidence")
print()
print("✓ Every trusted fragment has provenance")
print("✓ Every fragment resolves to ART-0001")
print("✓ Every fragment resolves to exact source location")
print("✓ Every evidence string verified")
print("✓ Source checksum verified")
print("✓ Trusted relationships remain trusted")
print("✓ Candidate relationships remain candidate")
print("✓ No LLM required for recollection")

print()
print("✓ CIF-01 PASS")
print("✓ CIF-02 PASS")
print("✓ REUSABLE ASSISTED RECOLLECTION PASS")

CEREBRO — Recollection Integrity
✓ KF-0001 → exact evidence
✓ KF-0002 → exact evidence
✓ KF-0003 → exact evidence

✓ Every trusted fragment has provenance
✓ Every fragment resolves to ART-0001
✓ Every fragment resolves to exact source location
✓ Every evidence string verified
✓ Source checksum verified
✓ Trusted relationships remain trusted
✓ Candidate relationships remain candidate
✓ No LLM required for recollection

✓ CIF-01 PASS
✓ CIF-02 PASS
✓ REUSABLE ASSISTED RECOLLECTION PASS


### 16E — Negative integrity test

In [70]:
from copy import deepcopy

tampered_model = deepcopy(
    knowledge_model
)

tampered_model["fragments"][0][
    "provenance"
]["source_sha256"] = "INVALID-CHECKSUM"

tamper_detected = False

try:

    recollection.recollect(
        "KF-0001",
        knowledge_model=tampered_model,
        registered_artifact=registered_artifact,
        source_text=source_text,
        candidate_relationships=candidate_relationships,
    )

except ValueError as exc:

    tamper_detected = True

    print("CEREBRO — Integrity Violation")
    print("=" * 60)
    print(exc)


assert tamper_detected is True

print()
print("✓ Provenance tampering detected")
print("✓ Recollection refused")
print("✓ FAIL-CLOSED RECOLLECTION PASS")

CEREBRO — Integrity Violation
Fragment source checksum does not match registered artifact.

✓ Provenance tampering detected
✓ Recollection refused
✓ FAIL-CLOSED RECOLLECTION PASS


### 17 — 06.5 Regression & Module Inventory

### 17A — Inventory reusable modules

In [72]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Reusable Capability Inventory
# ---------------------------------------------------------

expected_modules = [
    "inspection.py",
    "extraction.py",
    "metadata.py",
    "routing.py",
    "enrichment.py",
    "evaluation.py",
    "review.py",
    "registration.py",
    "chunking.py",
    "knowledge.py",
    "embedding.py",
    "relationships.py",
    "recollection.py",
]

src_dir = repo_root / "poc/src"

print("CEREBRO — Reusable Capability Inventory")
print("=" * 65)

missing_modules = []

for module_name in expected_modules:

    module_path = src_dir / module_name

    exists = module_path.exists()

    print(
        f"{'✓' if exists else '✗'} "
        f"{module_name}"
    )

    if not exists:
        missing_modules.append(module_name)

assert not missing_modules, (
    f"Missing modules: {missing_modules}"
)

print()
print(
    f"✓ {len(expected_modules)} reusable "
    "capabilities present"
)

CEREBRO — Reusable Capability Inventory
✓ inspection.py
✓ extraction.py
✓ metadata.py
✓ routing.py
✓ enrichment.py
✓ evaluation.py
✓ review.py
✓ registration.py
✓ chunking.py
✓ knowledge.py
✓ embedding.py
✓ relationships.py
✓ recollection.py

✓ 13 reusable capabilities present


### 17B — Verify frozen reference artifacts still exist

In [73]:
# ---------------------------------------------------------
# Frozen Steps 02–06 Outputs
# ---------------------------------------------------------

reference_files = {
    "registered_artifact":
        repo_root /
        "poc/data/processed/artifacts/ART-0001.json",

    "knowledge_model":
        repo_root /
        "poc/data/processed/knowledge/KM-0001.json",

    "semantic_candidates":
        repo_root /
        "poc/data/processed/knowledge/"
        "KM-0001-semantic-candidates.json",

    "experience_contract":
        repo_root /
        "poc/outputs/integration/"
        "CEREBRO-KNOWLEDGE-EXPERIENCE-v0.1.json",

    "raw_source":
        repo_root /
        "poc/data/raw/text/benchmark_001.txt",
}

print("CEREBRO — Frozen Reference Outputs")
print("=" * 65)

for name, path in reference_files.items():

    exists = path.exists()

    print(
        f"{'✓' if exists else '✗'} "
        f"{name:24} {path.relative_to(repo_root)}"
    )

    assert exists, f"Missing reference file: {path}"

print()
print("✓ Frozen Steps 02–06 outputs preserved")

CEREBRO — Frozen Reference Outputs
✓ registered_artifact      poc/data/processed/artifacts/ART-0001.json
✓ knowledge_model          poc/data/processed/knowledge/KM-0001.json
✓ semantic_candidates      poc/data/processed/knowledge/KM-0001-semantic-candidates.json
✓ experience_contract      poc/outputs/integration/CEREBRO-KNOWLEDGE-EXPERIENCE-v0.1.json
✓ raw_source               poc/data/raw/text/benchmark_001.txt

✓ Frozen Steps 02–06 outputs preserved


### 17C — Reload frozen reference data

In [74]:
# ---------------------------------------------------------
# Load Frozen Regression References
# ---------------------------------------------------------

with open(
    reference_files["registered_artifact"],
    encoding="utf-8"
) as f:
    frozen_artifact = json.load(f)

with open(
    reference_files["knowledge_model"],
    encoding="utf-8"
) as f:
    frozen_knowledge = json.load(f)

with open(
    reference_files["semantic_candidates"],
    encoding="utf-8"
) as f:
    frozen_candidates = json.load(f)

with open(
    reference_files["experience_contract"],
    encoding="utf-8"
) as f:
    frozen_experience = json.load(f)

frozen_source_text = (
    reference_files["raw_source"]
    .read_text(encoding="utf-8")
)

print("✓ Frozen regression data loaded")

✓ Frozen regression data loaded


### 17D — Validate canonical identities and source

In [75]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Stable Identity Regression
# ---------------------------------------------------------

EXPECTED_SHA256 = (
    "7694567acb26611218fd818ef2c12ec"
    "456e5ee94c5387804bfe5f36799aee832"
)

assert (
    registered_artifact["artifact_id"]
    == "ART-0001"
)

assert (
    registered_artifact["source"]["sha256"]
    == EXPECTED_SHA256
)

assert (
    reusable_inspection["file"]["sha256"]
    == EXPECTED_SHA256
)

assert (
    knowledge_model["knowledge_model_id"]
    == "KM-0001"
)

fragment_ids = [
    fragment["fragment_id"]
    for fragment in knowledge_model["fragments"]
]

assert fragment_ids == [
    "KF-0001",
    "KF-0002",
    "KF-0003",
]

relationship_ids = [
    relationship["relationship_id"]
    for relationship
    in knowledge_model["trusted_relationships"]
]

assert relationship_ids == [
    "REL-0001",
    "REL-0002",
    "REL-0003",
]

print("CEREBRO — Identity Regression")
print("=" * 65)

print("✓ ART-0001 preserved")
print("✓ KM-0001 preserved")
print("✓ KF-0001..KF-0003 preserved")
print("✓ REL-0001..REL-0003 preserved")
print("✓ Canonical SHA-256 preserved")

CEREBRO — Identity Regression
✓ ART-0001 preserved
✓ KM-0001 preserved
✓ KF-0001..KF-0003 preserved
✓ REL-0001..REL-0003 preserved
✓ Canonical SHA-256 preserved


### 17E — Full provenance regression

In [76]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Provenance Regression
# ---------------------------------------------------------

for fragment in knowledge_model["fragments"]:

    assert fragment["artifact_id"] == "ART-0001"

    assert (
        fragment["provenance"]["source_artifact_id"]
        == "ART-0001"
    )

    assert (
        fragment["provenance"]["source_sha256"]
        == EXPECTED_SHA256
    )

    location = fragment["source_location"]

    evidence = source_text[
        location["start_char"]:
        location["end_char"]
    ]

    assert evidence == fragment["content"]

print("CEREBRO — Provenance Regression")
print("=" * 65)

print("✓ Every KF retains artifact provenance")
print("✓ Every KF retains source checksum")
print("✓ Every KF retains exact source location")
print("✓ Every KF reconstructs exact evidence")

print()
print("✓ CIF-01 PASS")
print("✓ CIF-02 PASS")

CEREBRO — Provenance Regression
✓ Every KF retains artifact provenance
✓ Every KF retains source checksum
✓ Every KF retains exact source location
✓ Every KF reconstructs exact evidence

✓ CIF-01 PASS
✓ CIF-02 PASS


### 17F — Trust-state regression

In [77]:
# ---------------------------------------------------------
# CEREBRO 06.5 — Trust-State Regression
# ---------------------------------------------------------

assert all(
    relationship["status"] == "trusted"
    for relationship
    in knowledge_model["trusted_relationships"]
)

assert all(
    candidate["status"] == "candidate"
    for candidate
    in candidate_relationships
)

assert all(
    candidate["human_validated"] is False
    for candidate
    in candidate_relationships
)

assert (
    approved_artifact_metadata["human_reviewed"]
    is True
)

assert (
    approved_artifact_metadata["human_approved"]
    is True
)

assert registered_artifact["status"] == "REGISTERED"

print("CEREBRO — Trust-State Regression")
print("=" * 65)

print("✓ AI suggestions require human review")
print("✓ Registered artifact requires approval")
print("✓ Controlled relationships remain trusted")
print("✓ Machine-discovered relationships remain candidate")
print("✓ No silent trust promotion")

CEREBRO — Trust-State Regression
✓ AI suggestions require human review
✓ Registered artifact requires approval
✓ Controlled relationships remain trusted
✓ Machine-discovered relationships remain candidate
✓ No silent trust promotion


### 17G — End-to-end recollection regression

In [78]:
# ---------------------------------------------------------
# CEREBRO 06.5 — End-to-End Regression
# ---------------------------------------------------------

for fragment_id in [
    "KF-0001",
    "KF-0002",
    "KF-0003",
]:

    result = recollection.recollect(
        fragment_id,
        knowledge_model=knowledge_model,
        registered_artifact=registered_artifact,
        source_text=source_text,
        candidate_relationships=candidate_relationships,
    )

    assert result["status"] == "RECOLLECTED"
    assert result["integrity"]["source_resolved"]
    assert result["integrity"]["checksum_verified"]
    assert result["integrity"]["evidence_verified"]


print("CEREBRO — 06.5 FINAL REGRESSION")
print("=" * 65)

print("✓ Inspection")
print("✓ Extraction")
print("✓ Deterministic metadata")
print("✓ AI routing")
print("✓ AI enrichment")
print("✓ Evaluation")
print("✓ Human review")
print("✓ Registration")
print("✓ Knowledge construction")
print("✓ Provenance")
print("✓ Embeddings")
print("✓ Candidate relationships")
print("✓ Assisted recollection")
print("✓ CIF-01")
print("✓ CIF-02")

print()
print("======================================")
print("✓ CEREBRO STEP 06.5 REGRESSION PASS")
print("======================================")

CEREBRO — 06.5 FINAL REGRESSION
✓ Inspection
✓ Extraction
✓ Deterministic metadata
✓ AI routing
✓ AI enrichment
✓ Evaluation
✓ Human review
✓ Registration
✓ Knowledge construction
✓ Provenance
✓ Embeddings
✓ Candidate relationships
✓ Assisted recollection
✓ CIF-01
✓ CIF-02

✓ CEREBRO STEP 06.5 REGRESSION PASS
